# 16 · Gradient-Matched Temporal Label Reliability — **pre-GPU only**

Hiện thực `FINAL_PLAN_V3_3_LABEL_RELIABILITY.md`. Ghi chép quyết định:
`REVIEW_PLAN_V3_3_FINAL.md` (5/5 thắc mắc đã chốt) và `REVIEW_PLAN_V3_2_GRADIENT_MATCHED.md`
(§3-bis: lỗi CLS ở nb14/15).

**Notebook này KHÔNG train.** Nó chạy đúng bốn bước tiền-GPU của plan §13:

```
0a  raw labels + coverage + temporal manifest + concentration   (C2, C3, C4)
0b  exact initial ASL logit-gradient share + beta_c             (C8)
0c  200-step smoke: implementation / runtime / stochasticity     (C9)
0d  deterministic calibration diagnostics                        (C8 + C9)
```

Lý do dừng ở đây: **§4.4 của plan có thể kết thúc thí nghiệm ngay tại C4.** Nếu
data-sufficiency gate fail thì `R2` mất primary status và **không** được sửa manifest rồi
tiếp tục. Viết train loop trước khi biết là lãng phí.

## Ba arm (khoá)

| arm | ordinary blank | contested blank | mọi ô khác |
|---|---:|---:|---|
| `R0` historical | 1 | 1 | 1 |
| `R1GM` uniform control | `β_c` | `β_c` | 1 |
| `R2` temporal targeting | 1 | **η = 0.10** | 1 |

`β_c` **suy ra từ** `η` (§1 plan) nên control luôn tự khớp initial logit-gradient mass của
chính mức can thiệp đang dùng:

$$\beta_c=\frac{\eta\,G_{\text{cont},c}+G_{\text{ord},c}}{G_{\text{cont},c}+G_{\text{ord},c}}$$

`NonTarget-10` tự động có `β = 1` vì `G_cont = 0`.

## Bốn chỗ khác nb14/15 — đọc kỹ

**1. Patch-only.** `patches = backbone.layernorm(h)[:, 1:]` — **bỏ CLS**. nb14/15 đưa cả CLS
vào ML-Decoder, lệch với H5 (train trên 1,369 patch token) và với nb11/13. Xem
`REVIEW_PLAN_V3_2` §3-bis.

**2. Identity gate ba tầng** so với `ArmModel(plain=True)` **nguyên bản nb13**, code path độc
lập, không dùng chung module. Cổng `G2` của nb14 so hai instance của **cùng** implementation
nên không bắt được lỗi CLS.

**3. Partial fine-tuning thật.** Blocks 9–11 **trainable**, bản sao riêng mỗi arm. Không notebook
nào trước đây train block backbone.

**4. Chuẩn hoá loss bằng hằng số** `B_actual × C`, **không** chia `Σw` — nếu chia `Σw` thì 10 nhãn
non-target nhận gradient lệch ~20% giữa các arm chỉ vì mẫu số khác.

## Không dùng

```
report text · expert relabel · hard relabel · pair-consistency loss · EHR · WASL/DBL
```

## C0 · Config khoá (prereg)

In [1]:
import os, re, json, math, time, copy, pickle, random, shutil, tarfile, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')            # 04b LP-FT
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')   # 06b head
SC_DIR    = os.path.join(DATA_DIR, 'label_reliability')
for _d in (SC_DIR, f'{SC_DIR}/metrics', f'{SC_DIR}/checkpoints', f'{SC_DIR}/snapshots'):
    os.makedirs(_d, exist_ok=True)
PROJECT, CXR = 'mimic-ehr-v310', 'physionet-data.mimic_cxr_jpg'      # nguon nhan tho
RAW_PQ = os.path.join(OUT_DIR, 'cohort_raw_labels.parquet')          # FILE MOI, khong ghi de

# ---------------- KHOA: mot seed moi run ----------------
SEED = 42                       # doi thanh 1337 / 2024 cho seed xac nhan (§11 plan)

# ---------------- KHOA: arm & weight policy ----------------
ARMS  = ['R0', 'R1GM', 'R2']
ETA   = 0.10                    # weight cua contested blank trong R2
RARE3 = ['Pleural_Other', 'Fracture', 'Lung_Lesion']                 # khoa tu class_freq.pkl
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']

# ---------------- KHOA: temporal manifest ----------------
WINDOW_DAYS   = 7               # +/- 7 ngay, persistence prior cho structural label
ANCHOR_RAW    = 1               # chi raw +1 lam anchor; KHONG dung raw -1
REQUIRE_SAME_VIEW = True
REQUIRE_SAME_ENCOUNTER = False  # encounter la sensitivity, khong phai primary filter (§4.2)

# ---------------- KHOA: data sufficiency gate (§4.4) ----------------
GATE_CELLS_PER_CLASS    = 500
GATE_SUBJECTS_PER_CLASS = 200
GATE_CELLS_TOTAL        = 1000
GATE_SUBJECTS_TOTAL     = 500
GATE_TOP1PCT_SHARE      = 0.15  # top 1% subject <= 15% contested initial gradient mass

# ---------------- KHOA: optimization ----------------
EPOCHS, EFF = 8, 256
LR_BLOCKS, LR_HEAD = 1e-5, 1e-5            # blocks 9-11 + backbone.LN | H5 norm + decoder
CLIP_BLOCKS, CLIP_HEAD = 5.0, 1.0
WD = 1e-2                                   # chi ndim>1; bias & LayerNorm wd=0
WARMUP_FRAC = 0.05
LAMBDA_SP = 0.0                             # §6.1: BO L2-SP, khong claim stabilization
HEAD_WARMUP_EPOCHS = 1                      # ep1: train blocks+backbone.LN, freeze H5 norm+mld

# ---------------- KHOA: ASL (y het nb11-15) ----------------
ASL_GP, ASL_GN, ASL_CLIP, ASL_EPS = 1, 4, 0.05, 1e-8

# ---------------- KHOA: eval ----------------
W_F, W_L   = 0.7, 0.3
MACRO_TOL  = 0.0015
N_BOOT     = 2000               # cluster = subject_id
IDENT_TOL  = 1e-6               # nguong identity gate 3 tang

PREREG = dict(
    plan='FINAL_PLAN_V3_3_LABEL_RELIABILITY.md',
    review='REVIEW_PLAN_V3_3_FINAL.md (5/5 chot) + REVIEW_PLAN_V3_2 §3-bis (loi CLS)',
    scope='PRE-GPU ONLY: 0a coverage+manifest, 0b gradient share+beta, 0c smoke, 0d calibration',
    arms=ARMS, eta=ETA, rare3=RARE3, worst3_fixed=WORST3_FIXED,
    beta_rule='beta_c = (eta*G_cont + G_ord)/(G_cont + G_ord), do tai INIT, logit-gradient level',
    claim_locked='gradient matched at the output/logit level at initialization',
    claim_forbidden='parameter-gradient matched throughout training',
    manifest=dict(window_days=WINDOW_DAYS, anchor='raw +1 only', nearest_anchor_per_cell=True,
                  same_view=REQUIRE_SAME_VIEW, same_encounter_primary=REQUIRE_SAME_ENCOUNTER,
                  adjacency_required=False, train_only=True),
    fracture_rule=('bidirectional trong primary; BAT BUOC bao cao D2 tach theo huong. '
                   'Neu gain chi tu future-positive anchor -> chi duoc goi "temporal association", '
                   'KHONG duoc claim sua omission. Backward-only la sensitivity, khong thay primary.'),
    sufficiency_gate=dict(cells_per_class=GATE_CELLS_PER_CLASS, subjects_per_class=GATE_SUBJECTS_PER_CLASS,
                          cells_total=GATE_CELLS_TOTAL, subjects_total=GATE_SUBJECTS_TOTAL,
                          top1pct_gradient_share=GATE_TOP1PCT_SHARE,
                          on_fail='R2 mat primary status; KHONG sua manifest roi tiep tuc'),
    loss_norm='L = sum(w*l) / (B_actual * C)  — HANG SO, khong chia sum(w)',
    token='patch-only: backbone.layernorm(h)[:, 1:]  (BO CLS)',
    tail='blocks 9-11 (0-indexed) trainable, ban sao rieng moi arm; blocks 0-8 frozen shared',
    layernorms='backbone.layernorm -> tail group, trainable tu ep1 | H5 norm -> head group, frozen ep1',
    l2sp=LAMBDA_SP, epochs=EPOCHS, seed=SEED, effective_batch=EFF,
    lr=dict(blocks=LR_BLOCKS, head=LR_HEAD), clip=dict(blocks=CLIP_BLOCKS, head=CLIP_HEAD), wd=WD,
    hard_failure_only=['NaN/Inf', 'shape sai', 'blocks 0-8 bi doi', 'blocks 9-11 khong update',
                       'optimizer/scaler hong', 'output tro thanh hang so do loi so hoc'],
    drift_policy='DIAGNOSTIC ONLY — khong dung de dung run hoac loai checkpoint (§7)',
    seed_staging=('seed 42 truoc de validate implementation; sau do chay 1337 va 2024 BAT KE dau '
                  'cua delta seed 42 (§11 — tranh selection bias)'),
    eta_limitation='D2 ~ 0 chi chung minh targeting khong huu ich TAI eta=0.10',
    eval='historical frozen labels la primary; known-only la diagnostic only')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json', 'w'), indent=1)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU')
if dev == 'cuda': print(f'  VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
print(f'SEED {SEED} | arms {ARMS} | eta {ETA} | window +/-{WINDOW_DAYS}d | epochs {EPOCHS}')
print('PREREG khoa ->', f'{SC_DIR}/preregistration.json')
print('\nSCOPE: notebook nay KHONG train. Dung sau C9 (smoke) de doc so 0a/0b.')

Mounted at /content/drive
device: cuda | NVIDIA RTX PRO 6000 Blackwell Server Edition
  VRAM 102 GB
SEED 42 | arms ['R0', 'R1GM', 'R2'] | eta 0.1 | window +/-7d | epochs 8
PREREG khoa -> /content/drive/MyDrive/MIMIC_CXR/planB_out/label_reliability/preregistration.json

SCOPE: notebook nay KHONG train. Dung sau C9 (smoke) de doc so 0a/0b.


In [2]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san o {LOCAL_IMG}; khong co SHARD_DIR -> bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong -> can data'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards: {len(tars)} | to stage: {len(todo)} | da co {_have:,} anh')
    for f in tqdm(todo, desc='staging shards', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

shards: 347 | to stage: 347 | da co 0 anh


staging shards:   0%|          | 0/347 [00:00<?, ?shard/s]

/tmp/ipykernel_3923/3922000840.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)


local images: 346578


## C1 · Cohort + **nhãn thô bốn trạng thái**

Không dùng EHR nên C1 gọn hơn nb11–15 nhiều: chỉ cần nhãn, split, tier, view manifest.

Nhãn thô `{+1, 0, -1, blank}` **không còn** trong bất kứ file nào (01·C12 áp U-Ones rồi
`fillna(0)` **trước khi** ghi parquet). Phải query BigQuery một lần rồi cache ra **file mới**
`cohort_v2/cohort_raw_labels.parquet` — không ghi đè file cũ.

Assert bắt buộc: `U-Ones(raw) == nhãn cohort hiện có`, từng ô.

In [3]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS); assert C == 13
ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split', 'setting'] + LABELS)
       .sort_values('study_id').reset_index(drop=True))
N = len(ehr)
Y_STUDY = ehr[LABELS].to_numpy('float32')
STUDY_ROW = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}
SPLIT = ehr.split.to_numpy()
print(f'cohort {N:,} study | split {dict(zip(*np.unique(SPLIT, return_counts=True)))}')

# ---- tier + Rare-3 + NonTarget-10, KHOA tu class_freq.pkl (khong tinh lai) ----
cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TRAIN = int(cf['num_train'])
TIER = np.where(N_POS_TRAIN/N_TRAIN < 0.02, 'tail',
                np.where(N_POS_TRAIN/N_TRAIN < 0.10, 'medium', 'head'))
RARE3_IDX = np.sort(np.argsort(N_POS_TRAIN, kind='stable')[:3])
assert set(LABELS[j] for j in RARE3_IDX) == set(RARE3), \
    f'Rare-3 lech: {[LABELS[j] for j in RARE3_IDX]} vs {RARE3}'
RARE3_IDX = np.array([LABELS.index(l) for l in RARE3])          # theo dung thu tu RARE3
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
W3_IDX = np.array([LABELS.index(l) for l in WORST3_FIXED])
TAIL_IDX = np.where(TIER == 'tail')[0]
print(f'Rare-3 (khoa): {[(l, int(N_POS_TRAIN[LABELS.index(l)])) for l in RARE3]}')
print(f'NonTarget-10 : {len(NONTGT_IDX)} nhan | Worst-3 fixed: {WORST3_FIXED}')

# ---- nhan tho 4 trang thai: cache ra FILE MOI ----
if os.path.exists(RAW_PQ):
    rawdf = pd.read_parquet(RAW_PQ); print(f'raw labels <- cache {RAW_PQ}')
else:
    from google.cloud import bigquery
    try:
        from google.colab import auth
        !gcloud auth application-default login --no-launch-browser
    except Exception:
        pass
    print('query BigQuery ...')
    chex = bigquery.Client(project=PROJECT, location='US').query(
        f'SELECT * FROM `{CXR}.chexpert`').to_dataframe()
    _nm = lambda s: re.sub(r'[^a-z]', '', str(s).lower())
    _col = {_nm(c): c for c in chex.columns}
    miss = [l for l in LABELS if _nm(l) not in _col]
    assert not miss, f'khong map duoc cot: {miss}'
    rawdf = chex.drop_duplicates('study_id').set_index('study_id')[[_col[_nm(l)] for l in LABELS]]
    rawdf.columns = [f'raw_{l}' for l in LABELS]
    rawdf = rawdf.reset_index()
    rawdf.to_parquet(RAW_PQ, index=False)
    print(f'raw labels -> {RAW_PQ}  ({len(rawdf):,} rows)  [FILE MOI, khong ghi de]')

RAW = (rawdf.drop_duplicates('study_id').set_index('study_id')
       .reindex(ehr.study_id.to_numpy())[[f'raw_{l}' for l in LABELS]].to_numpy('float64'))
assert RAW.shape == (N, C)

# ---- ASSERT: U-Ones(raw) phai tai tao dung nhan cohort ----
_y = np.where(np.isnan(RAW), 0.0, np.where(RAW == -1, 1.0, RAW)).astype('float32')
_bad = int((_y != Y_STUDY).sum())
print(f'\nU-Ones(raw) == cohort labels ? {_bad == 0}  ({_bad} o lech)')
assert _bad == 0, 'nhan tho KHONG tai tao duoc nhan cohort -> dung chay tiep'

# ---- state code: 1=+1, 0=explicit 0, -1=uncertain, 2=blank ----
STATE = np.where(np.isnan(RAW), 2, RAW).astype('int8')
IS_BLANK = STATE == 2
print(f'trang thai (toan cohort): +1 {int((STATE==1).sum()):>9,} | 0 {int((STATE==0).sum()):>9,} '
      f'| -1 {int((STATE==-1).sum()):>9,} | blank {int(IS_BLANK.sum()):>9,}')

# ---- view manifest (giong nb11-15) ----
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id', 'subject_id', 'split']], on='study_id', how='inner')
_before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
          ].sort_values('dicom_id').reset_index(drop=True)
assert len(man) and (_before-len(man))/_before < 0.01, f'{_before-len(man)} anh thieu -> stage anh'
man['study_row'] = man.study_id.map(STUDY_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy() == s)[0] for s in ('train', 'val', 'test')}
print(f'anh theo split: { {s: int(len(IDX[s])) for s in IDX} }')

# ---- view signature moi study: F / L / FL ----
VSIG = (man.groupby('study_id')['view_group']
        .apply(lambda s: ''.join(sorted({x[0].upper() for x in s}))))
VSIG = VSIG.reindex(ehr.study_id.to_numpy()).fillna('').to_numpy()
print('view signature:', dict(zip(*np.unique(VSIG, return_counts=True))))
print('\nNOTE: TEST khong bao gio dung trong notebook nay.')

cohort 211,130 study | split {'test': np.int64(31669), 'train': np.int64(147791), 'val': np.int64(31670)}
Rare-3 (khoa): [('Pleural_Other', 1763), ('Fracture', 3212), ('Lung_Lesion', 4761)]
NonTarget-10 : 10 nhan | Worst-3 fixed: ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']
raw labels <- cache /content/drive/MyDrive/MIMIC_CXR/planB_out/cohort_v2/cohort_raw_labels.parquet

U-Ones(raw) == cohort labels ? True  (0 o lech)
trang thai (toan cohort): +1   332,561 | 0   141,201 | -1    70,017 | blank 2,200,911
anh theo split: {'train': 242497, 'val': 51880, 'test': 52201}
view signature: {'F': np.int64(109052), 'FL': np.int64(101712), 'L': np.int64(366)}

NOTE: TEST khong bao gio dung trong notebook nay.


## C2 · Coverage audit + timestamp audit (**0a-1**)

In **số tuyệt đối** `+1 / 0 / -1 / blank` cho 13 nhãn × 3 split (§5.1 plan).

Cột `n(raw==0)` là chỗ đáng nhìn nhất: nó là lý do V2 (masking) bị chặn. V3.3 **không mask**
nên nó không còn chặn, nhưng vẫn cho biết `Endpoint K` sẽ nhỏ và lệch prevalence tới mức nào.

Timestamp audit (§5.4 → §12): đếm `missing`, `00:00:00`, `duplicate subject+time`. **Không**
giả định `00:00` luôn là date-only — chỉ đếm, rule quyết định sau khi thấy số.

In [4]:
rows = []
for j, l in enumerate(LABELS):
    for sp in ('train', 'val', 'test'):
        m = SPLIT == sp
        v = STATE[m, j]
        n = int(m.sum())
        rows.append(dict(label=l, tier=TIER[j], rare3=l in RARE3, split=sp, n_study=n,
                         n_pos=int((v == 1).sum()), n_neg0=int((v == 0).sum()),
                         n_unc=int((v == -1).sum()), n_blank=int((v == 2).sum()),
                         known_rate=float(((v == 1) | (v == 0)).mean()),
                         prev_definite_pos=float((v == 1).mean()),
                         prev_definite_neg=float((v == 0).mean())))
CV = pd.DataFrame(rows)
CV.to_csv(f'{SC_DIR}/metrics/raw_label_counts.csv', index=False)
pd.set_option('display.width', 220)
print('=== TRAIN (§5.1) ===')
print(CV[CV.split == 'train'].drop(columns=['split', 'n_study']).to_string(
      index=False, float_format=lambda v: f'{v:.4f}'))

print('\n=== n(raw==0) — known negative, theo split ===')
print(CV.pivot_table(index='label', columns='split', values='n_neg0')
        .reindex(LABELS)[['train', 'val', 'test']].to_string())
_r3 = CV[(CV.split == 'train') & CV.rare3]
print('\nRare-3 TRAIN known negative:',
      {r.label: int(r.n_neg0) for r in _r3.itertuples()})
print('  (V2 bi chan vi so nay < 1000; V3.3 KHONG mask nen khong con chan.')
print('   Van ghi lai vi no quyet dinh co mau va prevalence cua Endpoint K.)')

# ---------------- timestamp audit ----------------
_base = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'),
                        columns=['study_id', 'subject_id', 'index_time'])
_base = _base.drop_duplicates('study_id').set_index('study_id').reindex(ehr.study_id.to_numpy())
T = pd.to_datetime(_base['index_time'])
TSEC = T.astype('int64').to_numpy() / 1e9                       # giay, NaT -> rat am
n_missing = int(T.isna().sum())
_midnight = ((T.dt.hour == 0) & (T.dt.minute == 0) & (T.dt.second == 0)).fillna(False)
_dupkey = pd.Series(ehr.subject_id.astype(str).to_numpy() + '|' + T.astype(str).to_numpy())
n_dup = int(_dupkey.duplicated(keep=False).sum())
TS_OK = (~T.isna()).to_numpy()
print(f'\n=== timestamp audit (§5.4) ===')
print(f'  missing index_time        {n_missing:>8,}  ({n_missing/N:.3%})')
print(f'  time == 00:00:00          {int(_midnight.sum()):>8,}  ({_midnight.mean():.3%})')
print(f'  duplicate subject+time    {n_dup:>8,}  ({n_dup/N:.3%})')
json.dump(dict(n_study=N, missing_index_time=n_missing, midnight=int(_midnight.sum()),
               duplicate_subject_time=n_dup, rule='CHUA CHOT — quyet dinh sau khi doc so nay'),
          open(f'{SC_DIR}/metrics/timestamp_audit.json', 'w'), indent=1)
print(f'''
  RULE CHUA CHOT (§12 plan). Doc so tren roi quyet dinh:
    - midnight < ~1%  -> loai chung khoi manifest (an toan, mat it cell)
    - midnight nhieu  -> GIU, vi 00:00 co the la gio chup that; ghi nhan la nguon nhieu
  Hien tai TS_OK = (index_time khong NaT). Doi lai o C4 neu muon chat hon.''')

=== TRAIN (§5.1) ===
                     label   tier  rare3  n_pos  n_neg0  n_unc  n_blank  known_rate  prev_definite_pos  prev_definite_neg
               Atelectasis   head  False  30181     997   6791   109822      0.2110             0.2042             0.0067
              Cardiomegaly   head  False  29490   10012   3815   104474      0.2673             0.1995             0.0677
             Consolidation medium  False   7117    5189   2817   132668      0.0833             0.0482             0.0351
                     Edema   head  False  17828   16262   8725   104976      0.2307             0.1206             0.1100
Enlarged_Cardiomediastinum medium  False   4767    3324   6081   133619      0.0547             0.0323             0.0225
                  Fracture medium   True   2865     586    347   143993      0.0234             0.0194             0.0040
               Lung_Lesion medium   True   4052     512    709   142518      0.0309             0.0274             0.0035
   

## C3 · Encounter metadata

`cur_hadm` / `cur_edstay` nằm ở **`cohort_base.parquet`** (01·C18 `base_cols`), **không** ở
`cohort_study_level.parquet` mà nb11–15 đang load. Phải merge thêm.

ICU `stay_id` **không có** trong cohort ⇒ điều kiện ICU của các bản plan cũ rơi về `hadm_id`.

Encounter **không** phải primary filter (§4.2) — chỉ dùng để báo cáo sensitivity.

In [5]:
_bcols = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet')).columns.tolist()
print('cot co trong cohort_base.parquet:', _bcols)
_want = [c for c in ('cur_hadm', 'cur_edstay') if c in _bcols]
assert _want, 'khong thay cur_hadm/cur_edstay -> khong the bao cao sensitivity theo encounter'
enc = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_base.parquet'),
                       columns=['study_id'] + _want)
       .drop_duplicates('study_id').set_index('study_id').reindex(ehr.study_id.to_numpy()))
HADM = enc['cur_hadm'].to_numpy() if 'cur_hadm' in enc else np.full(N, np.nan)
EDST = enc['cur_edstay'].to_numpy() if 'cur_edstay' in enc else np.full(N, np.nan)
print(f'  cur_hadm   co gia tri {np.sum(pd.notna(HADM)):>9,} / {N:,}  ({pd.notna(HADM).mean():.1%})')
print(f'  cur_edstay co gia tri {np.sum(pd.notna(EDST)):>9,} / {N:,}  ({pd.notna(EDST).mean():.1%})')
print('  ICU stay_id: KHONG co trong cohort -> khong dung')

cot co trong cohort_base.parquet: ['study_id', 'subject_id', 'index_time', 'setting', 'cur_hadm', 'cur_edstay', 'n_images', 'n_usable_view', 'n_frontal', 'n_lateral', 'n_view_ok', 'split', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged_Cardiomediastinum', 'Fracture', 'Lung_Lesion', 'Lung_Opacity', 'No_Finding', 'Pleural_Effusion', 'Pleural_Other', 'Pneumonia', 'Pneumothorax']
  cur_hadm   co gia tri   110,820 / 211,130  (52.5%)
  cur_edstay co gia tri    81,662 / 211,130  (38.7%)
  ICU stay_id: KHONG co trong cohort -> khong dung


## C4 · Temporal manifest ±7 ngày + sufficiency gate (**0a-2**)

Một ô `(study, label)` là **contested** khi tồn tại anchor thoả (§4.1 plan):

```
cung subject_id
raw cua chinh study  = blank
raw cua anchor       = +1        (KHONG dung raw -1)
0 < |delta_t| <= 7 ngay
cung view signature
timestamp dung/tin cay ca hai ben
```

**Không** yêu cầu chronological-adjacent — quét **mọi** study của bệnh nhân trong cửa sổ, rồi
lấy **anchor gần nhất** cho mỗi ô blank, đánh dấu **một lần**. Nhờ vậy khai thác hết bằng chứng
dọc thời gian mà không tạo mẫu bậc hai.

Chỉ dùng **TRAIN** để tạo weight. Cửa sổ 7 ngày là **persistence prior** cho tổn thương cấu
trúc, không phải bằng chứng hai study có cùng trạng thái bệnh.

### `Fracture` — quy tắc khoá riêng

Giữ **hai chiều** trong primary, nhưng bắt buộc báo cáo `D2` tách theo hướng:

| hướng | diễn giải cho phép |
|---|---|
| anchor **trước** blank | persistence/omission — tương đối mạnh |
| blank → anchor **sau** | có thể là gãy **mới** — yếu hơn |

Nếu gain của `Fracture` **chỉ** đến từ future-positive anchor: chỉ được gọi **temporal
association**, **không** được claim sửa omission.

In [6]:
IS_TRAIN = SPLIT == 'train'
SUBJ = ehr.subject_id.to_numpy()
DAY = 86400.0
WIN = WINDOW_DAYS * DAY

CONTESTED = np.zeros((N, C), bool)                 # o (study, label) bi tranh chap
GAP_H     = np.full((N, C), np.nan, 'float32')     # gio, tri tuyet doi
ANCH_ROW  = np.full((N, C), -1, 'int64')           # study row cua anchor gan nhat
FUNNEL, DETAIL = [], []

_ord = np.argsort(SUBJ, kind='stable')
_bnd = np.r_[0, np.flatnonzero(np.diff(SUBJ[_ord])) + 1, len(_ord)]
_groups = [_ord[_bnd[k]:_bnd[k+1]] for k in range(len(_bnd)-1)]
print(f'{len(_groups):,} subject | quet cua so +/-{WINDOW_DAYS}d cho {len(RARE3)} nhan Rare-3\n')

for l in RARE3:
    j = LABELS.index(l)
    n_cand = n_ts = n_win = n_view = 0
    for g in _groups:
        g = g[IS_TRAIN[g]]
        if len(g) < 2: continue
        st = STATE[g, j]
        ip, ib = g[st == 1], g[st == 2]
        if not len(ip) or not len(ib): continue
        n_cand += len(ib)
        okp, okb = TS_OK[ip], TS_OK[ib]
        ip, ib = ip[okp], ib[okb]
        if not len(ip) or not len(ib): continue
        n_ts += len(ib)
        d = np.abs(TSEC[ib][:, None] - TSEC[ip][None, :])          # [n_blank, n_pos]
        ok = (d > 0) & (d <= WIN)
        n_win += int(ok.any(1).sum())
        if REQUIRE_SAME_VIEW:
            ok &= (VSIG[ib][:, None] == VSIG[ip][None, :])
        hit = ok.any(1)
        n_view += int(hit.sum())
        if not hit.any(): continue
        dm = np.where(ok, d, np.inf)
        k = dm.argmin(1)
        for bi, ki, h in zip(ib[hit], k[hit], hit.nonzero()[0]):
            a = ip[ki]; gap = float(dm[h, ki])
            CONTESTED[bi, j] = True; GAP_H[bi, j] = gap/3600.0; ANCH_ROW[bi, j] = a
            DETAIL.append((l, int(SUBJ[bi]), int(bi), int(a), gap/3600.0,
                           'past' if TSEC[a] < TSEC[bi] else 'future',
                           bool(pd.notna(HADM[bi]) and HADM[bi] == HADM[a]),
                           bool(pd.notna(EDST[bi]) and EDST[bi] == EDST[a])))
    n_fin = int(CONTESTED[:, j].sum())
    FUNNEL.append(dict(label=l, blank_with_pos_subject=n_cand, plus_timestamp_ok=n_ts,
                       plus_within_window=n_win, plus_same_view=n_view, final=n_fin,
                       n_blank_train=int((IS_BLANK[:, j] & IS_TRAIN).sum()),
                       frac_of_blank=n_fin/max(int((IS_BLANK[:, j] & IS_TRAIN).sum()), 1)))
FUN = pd.DataFrame(FUNNEL); FUN.to_csv(f'{SC_DIR}/metrics/temporal_manifest_funnel.csv', index=False)
print('=== FUNNEL (§5.2) ===')
print(FUN.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

DET = pd.DataFrame(DETAIL, columns=['label','subject_id','study_row','anchor_row','gap_h',
                                    'direction','same_hadm','same_edstay'])
DET['gap_bin'] = pd.cut(DET.gap_h, [0, 24, 72, 24*7],
                        labels=['<=24h', '24-72h', '72h-7d'], include_lowest=True)
DET.to_parquet(f'{SC_DIR}/metrics/contested_cells.parquet', index=False)

print('\n=== CONCENTRATION + huong + gap (§4.3, §5.5) ===')
crows = []
for l in RARE3:
    d = DET[DET.label == l]
    per = d.groupby('subject_id').size().sort_values(ascending=False)
    top1 = per.iloc[:max(1, int(np.ceil(0.01*len(per))))].sum() / max(len(d), 1)
    crows.append(dict(label=l, cells=len(d), studies=d.study_row.nunique(),
                      subjects=len(per), median_cells_per_subject=float(per.median()) if len(per) else 0.0,
                      max_cells_one_subject=int(per.max()) if len(per) else 0,
                      top1pct_CELL_share=float(top1),
                      past=int((d.direction == 'past').sum()), future=int((d.direction == 'future').sum()),
                      **{f'gap_{b}': int((d.gap_bin == b).sum()) for b in ['<=24h','24-72h','72h-7d']},
                      same_hadm=int(d.same_hadm.sum()), same_edstay=int(d.same_edstay.sum())))
CON = pd.DataFrame(crows); CON.to_csv(f'{SC_DIR}/metrics/contested_cells_by_class.csv', index=False)
print(CON.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print('\n  LUU Y: top1pct_CELL_share la ty phan theo SO CELL. Gate §4.4 doi ty phan theo')
print('  GRADIENT MASS -> tinh o C8 sau khi co g_{k,c}. Con so o day chi de xem truoc.')

print('\n=== DATA SUFFICIENCY GATE (§4.4) — GRADIENT SHARE se kiem lai o C8 ===')
_tc, _ts_ = int(CON.cells.sum()), DET.subject_id.nunique()
_pass_cls = {r.label: dict(cells=r.cells >= GATE_CELLS_PER_CLASS,
                           subjects=r.subjects >= GATE_SUBJECTS_PER_CLASS) for r in CON.itertuples()}
for l, v in _pass_cls.items():
    r = CON[CON.label == l].iloc[0]
    print(f'  {l:<16} cells {int(r.cells):>6,} (>={GATE_CELLS_PER_CLASS}) {"OK" if v["cells"] else "FAIL":<4} | '
          f'subjects {int(r.subjects):>6,} (>={GATE_SUBJECTS_PER_CLASS}) {"OK" if v["subjects"] else "FAIL"}')
print(f'  Rare-3 TONG      cells {_tc:>6,} (>={GATE_CELLS_TOTAL}) '
      f'{"OK" if _tc >= GATE_CELLS_TOTAL else "FAIL":<4} | subjects {_ts_:>6,} '
      f'(>={GATE_SUBJECTS_TOTAL}) {"OK" if _ts_ >= GATE_SUBJECTS_TOTAL else "FAIL"}')
_gate_ok = all(all(v.values()) for v in _pass_cls.values()) and _tc >= GATE_CELLS_TOTAL and _ts_ >= GATE_SUBJECTS_TOTAL
print(f'''
  KET LUAN TAM ({"PASS" if _gate_ok else "FAIL"}) — con cho gradient-share o C8.
  Neu FAIL: theo §4.4, R2 MAT primary status. KHONG duoc sua manifest roi tiep tuc claim
  cung estimand. Sensitivity "cap <= 3 cells/subject" duoc phep nhung phai ghi exploratory.''')
np.save(f'{SC_DIR}/metrics/contested_mask.npy', CONTESTED)
json.dump(dict(gate_cells_subjects_pass=bool(_gate_ok), total_cells=_tc, total_subjects=int(_ts_)),
          open(f'{SC_DIR}/metrics/sufficiency_partial.json', 'w'), indent=1)

60,798 subject | quet cua so +/-7d cho 3 nhan Rare-3

=== FUNNEL (§5.2) ===
        label  blank_with_pos_subject  plus_timestamp_ok  plus_within_window  plus_same_view  final  n_blank_train  frac_of_blank
Pleural_Other                    9367               9367                9367            6617   6617         145945         0.0453
     Fracture                   10255              10255               10255            7430   7430         143993         0.0516
  Lung_Lesion                   16112              16112               16112           11191  11191         142518         0.0785

=== CONCENTRATION + huong + gap (§4.3, §5.5) ===
        label  cells  studies  subjects  median_cells_per_subject  max_cells_one_subject  top1pct_CELL_share  past  future  gap_<=24h  gap_24-72h  gap_72h-7d  same_hadm  same_edstay
Pleural_Other   6617     6617       732                    5.0000                    151              0.1135  2994    3623       6335         282           0       2430    

## C5 · Dataset + loader — **không EHR**

nb16 không dùng EHR (§1 plan) nên dataset gọn hơn nb11–15 rất nhiều: chỉ `image`, `y`,
`study_row`. Không `EmbeddingBag`, không ragged bag, không `set_id/rec_id/stat_id`.

`study_row` cần cho weight matrix — mỗi ảnh thừa hưởng weight của study nó thuộc về.

`num_workers=0` cho mọi eval/predict loader (memory: `dataloader-teardown-spam`).

In [7]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
print(f'MODEL_SIZE {MODEL_SIZE} | n_patch = {(MODEL_SIZE//14)**2} | +CLS = {(MODEL_SIZE//14)**2+1}')

class ImgDS(Dataset):
    """Anh + nhan study + study_row. KHONG co EHR."""
    def __init__(s, idx, tf): s.idx, s.tf = np.asarray(idx), tf
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        r = man.iloc[s.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.study_row)
        return dict(image=s.tf(im), y=torch.from_numpy(Y_STUDY[tr]), sr=tr)

def collate(bb):
    return dict(image=torch.stack([x['image'] for x in bb]),
                y=torch.stack([x['y'] for x in bb]),
                sr=torch.tensor([x['sr'] for x in bb], dtype=torch.int64))

def loader(split, tf, shuffle, bs, workers=8):
    return DataLoader(ImgDS(IDX[split], tf), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), persistent_workers=(workers > 0))

to_dev = lambda b: {k: v.to(dev, non_blocking=True) for k, v in b.items()}
_b = to_dev(collate([ImgDS(IDX['val'], eval_tf)[k] for k in range(4)]))
print('batch keys:', list(_b), '| image', tuple(_b['image'].shape), '| y', tuple(_b['y'].shape))
assert (Y_STUDY[_b['sr'].cpu().numpy()] == _b['y'].cpu().numpy()).all(), 'sr khong khop y'
print('dataset OK (khong EHR)')

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

MODEL_SIZE 518 | n_patch = 1369 | +CLS = 1370
batch keys: ['image', 'y', 'sr'] | image (4, 3, 518, 518) | y (4, 13)
dataset OK (khong EHR)


## C6 · Model — blocks 0–8 frozen shared · **9–11 trainable bản sao riêng** · H5 head patch-only

Ba nhóm tham số, khác hoàn toàn nb11–15:

| nhóm | module | thuộc | lr | clip | ep1 |
|---|---|---|---:|---:|---|
| shared frozen | `backbone.embeddings` + blocks 0–8 | dùng chung, `no_grad` | — | — | frozen |
| **tail** | blocks 9–11 + `backbone.layernorm` | **bản sao riêng mỗi arm** | `1e-5` | 5.0 | **train** |
| head | H5 `norm` + `mld` | bản sao riêng mỗi arm | `1e-5` | 1.0 | **frozen** |

`backbone.layernorm` thuộc nhóm **tail** (§5.2) — nó là LayerNorm cuối của backbone, không phải
của head. H5 `norm` là LayerNorm riêng của head, đóng băng ở ep1 cùng decoder.

### Patch-only — sửa lỗi CLS của nb14/15

```python
h = tail(z_shared)              # blocks 9,10,11 -> backbone.layernorm
logits = head(h[:, 1:])         # BO CLS  <-- nb14/15 thieu dong nay
```

Vì blocks 9–11 giờ **trainable** nên `forward_shared` chỉ chạy tới block 8 — rẻ hơn nb13
(chạy 0–8 no_grad) nhưng phần đuôi đắt hơn nhiều vì phải giữ activation cho backward qua 3
block × 1,370 token.

In [8]:
from transformers import AutoModel

TRAIN_FROM = 9              # 0-indexed: blocks 9,10,11 trainable (= block 10,11,12 1-indexed)
_probe = AutoModel.from_pretrained('microsoft/rad-dino')
N_BLOCKS = len(_probe.encoder.layer); D = _probe.config.hidden_size; del _probe
assert D == 768 and TRAIN_FROM < N_BLOCKS
N_PATCH = (MODEL_SIZE // 14) ** 2

assert os.path.exists(BB_CKPT), f'04b backbone khong thay: {BB_CKPT}'
_sd04b = torch.load(BB_CKPT, map_location='cpu', weights_only=False)['model']
BB_SD = {k[9:]: v for k, v in _sd04b.items() if k.startswith('backbone.')}
assert BB_SD, 'khong co backbone.* trong 04b checkpoint'

def load_backbone(device=dev):
    m = AutoModel.from_pretrained('microsoft/rad-dino')
    m.load_state_dict(BB_SD, strict=True)
    return m.to(device).eval()

# ---------------- shared frozen: embeddings + blocks 0..TRAIN_FROM-1 ----------------
backbone = load_backbone()
for p in backbone.parameters(): p.requires_grad_(False)
LAYERS = backbone.encoder.layer
def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

@torch.no_grad()
def forward_shared(x):
    h = backbone.embeddings(x)
    for i in range(TRAIN_FROM): h = _layer(h, i)
    return h.detach()

# ---------------- H5 head ----------------
import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw): _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None); return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True

assert os.path.exists(H5_CKPT), f'H5 head khong thay: {H5_CKPT}'
H5_SD = torch.load(H5_CKPT, map_location='cpu', weights_only=False)['model']
_sub = lambda p: {k[len(p):]: v for k, v in H5_SD.items() if k.startswith(p)}
assert _sub('norm.') and _sub('mld.'), f'H5 thieu norm.*/mld.*: {list(H5_SD)[:5]}'
print(f'H5 head: norm {len(_sub("norm."))} tensor | mld {len(_sub("mld."))} tensor')

# ---------------- per-arm modules ----------------
_TAIL_SRC = load_backbone('cpu')                # nguon deepcopy, tao 1 lan
for p in _TAIL_SRC.parameters(): p.requires_grad_(False)

class TailBackbone(nn.Module):
    """Blocks TRAIN_FROM..11 + backbone.layernorm — TRAINABLE, ban sao rieng moi arm."""
    def __init__(s):
        super().__init__()
        s.blocks = nn.ModuleList([copy.deepcopy(_TAIL_SRC.encoder.layer[i])
                                  for i in range(TRAIN_FROM, N_BLOCKS)])
        s.ln = copy.deepcopy(_TAIL_SRC.layernorm)
        for p in s.parameters(): p.requires_grad_(True)
    def forward(s, h):
        for b in s.blocks:
            o = b(h); h = o[0] if isinstance(o, (tuple, list)) else o
        return s.ln(h)

class H5Head(nn.Module):
    """H5 norm + ML-Decoder, patch-only input."""
    def __init__(s):
        super().__init__()
        s.norm = nn.LayerNorm(D)
        s.mld = MLDecoder(num_classes=C, initial_num_features=D, decoder_embedding=D, num_of_groups=C)
        s.norm.load_state_dict(_sub('norm.')); s.mld.load_state_dict(_sub('mld.'))
    def forward(s, patches): return s.mld(s.norm(patches))

class Arm(nn.Module):
    def __init__(s):
        super().__init__(); s.tail = TailBackbone(); s.head = H5Head()
    def forward(s, z_shared):
        h = s.tail(z_shared)
        assert h.shape[1] == N_PATCH + 1, f'token {h.shape[1]} != {N_PATCH+1}'
        return s.head(h[:, 1:])                      # <-- PATCH-ONLY, bo CLS

def build_arm(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return Arm().to(dev)

tail_params = lambda m: list(m.tail.parameters())
head_params = lambda m: list(m.head.parameters())

_A = build_arm()
_np = lambda ps: sum(p.numel() for p in ps)
print(f'\ntrainable/arm {_np(tail_params(_A))+_np(head_params(_A)):>12,}'
      f'  = tail {_np(tail_params(_A)):,} + head {_np(head_params(_A)):,}')
print(f'frozen shared  {_np(list(backbone.parameters())):>12,}  (dung chung {len(ARMS)} arm)')
print(f'snapshot/arm   ~{(_np(tail_params(_A))+_np(head_params(_A)))*4/1e6:.0f} MB')

# ---- G0: forward_shared + tail == backbone(x) stock (tai init) ----
_x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    _ref = backbone(pixel_values=_x).last_hidden_state
    _got = _A.tail(forward_shared(_x))
_d = (_ref - _got).abs().max().item()
print(f'\nG0 forward_shared+tail == backbone(x).last_hidden_state  max|diff| {_d:.2e} '
      f'{"OK" if _d < 1e-4 else "FAIL"}')
assert _d < 1e-4
print(f'   -> ban sao blocks {TRAIN_FROM}..{N_BLOCKS-1} + layernorm giu dung trong so 04b')

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

H5 head: norm 2 tensor | mld 19 tensor


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


trainable/arm   27,397,389  = tail 21,269,760 + head 6,127,629
frozen shared    86,580,480  (dung chung 3 arm)
snapshot/arm   ~110 MB

G0 forward_shared+tail == backbone(x).last_hidden_state  max|diff| 0.00e+00 OK
   -> ban sao blocks 9..11 + layernorm giu dung trong so 04b


## C7 · **Identity gate ba tầng** vs `ArmModel(plain=True)` của nb13 + stochasticity

Chốt ở `REVIEW_PLAN_V3_3_FINAL.md` §2·Q1. Năm điều kiện:

1. reference khởi tạo **trực tiếp từ code/class của nb13** — không viết lại `forward` cho nb16;
2. load backbone 04b và H5 `norm`/`mld` **độc lập** (instance riêng);
3. reference và candidate **không dùng chung** object/module nào;
4. `eval()`, FP32, cùng một batch, không augmentation;
5. so **ba tầng**, không chỉ logit.

| tầng | so cái gì | vì sao |
|---|---|---|
| **L1** | token sau **block 11**, *trước* final LN | lệch ở đây ⇒ lỗi ở bản sao block hoặc thứ tự chạy |
| **L2** | `backbone.layernorm(h)[:, 1:]` | lệch chỉ ở đây ⇒ lỗi ở final LN hoặc **chỗ cắt CLS** |
| **L3** | logits sau H5 head | lệch chỉ ở đây ⇒ lỗi ở `norm`/`mld` |

Ngưỡng `max|diff| ≤ 1e-6`. Nếu chỉ so L3 thì khi fail chỉ biết "có lệch", không biết ở đâu —
đó là lý do dùng ba tầng.

**Cổng `G2` của nb14 không bắt được lỗi CLS** vì nó so hai instance của *cùng* một class. Ở đây
reference đọc `h[:, 1:]` trong forward của **nb13**, candidate đọc `h[:, 1:]` trong forward của
nb16 — hai code path riêng. Nếu một bên quên `[:, 1:]` thì L2 và L3 sẽ khác.

Lấy L1/L2 bằng **forward hook** để `_r_forward_tail` giữ nguyên verbatim, không phải sửa nó.

### Stochasticity (§7 plan · 0c)

RAD-DINO/Dinov2 mặc định `dropout = 0` và `drop_path_rate = 0`. Cần **in ra** thay vì giả định:
nếu bằng 0 thì `train()` và `eval()` cho cùng output ⇒ **không** cần `sync_rng`, và mọi khác
biệt giữa arm chỉ đến từ weight.

In [9]:
# ================== REFERENCE: VERBATIM tu 13_dbl_weight_screen.ipynb ==================
# Instance backbone RIENG, khong dung chung voi `backbone` cua nb16.
_rb = load_backbone()
for p in _rb.parameters(): p.requires_grad_(False)
_rLAYERS = _rb.encoder.layer; _rn_blocks = len(_rLAYERS)
_rLATE = (9, 10)                              # nb13: LATE_1IDX=(10,11) -> 0-indexed (9,10)
_rSHARED_UPTO = min(_rLATE)                   # = 9, trung TRAIN_FROM
assert _rSHARED_UPTO == TRAIN_FROM

def _r_layer(h, i):                                                   # verbatim nb13
    o = _rLAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

def _r_forward_shared(x):                                             # verbatim nb13
    with torch.no_grad():
        h = _rb.embeddings(x)
        for i in range(_rSHARED_UPTO): h = _r_layer(h, i)
    return h.detach()

def _r_forward_tail(h, adapter_blocks=(), adapters=None, e=None):     # verbatim nb13
    first = min(adapter_blocks) if adapter_blocks else _rn_blocks
    for i in range(_rSHARED_UPTO, _rn_blocks):
        with (torch.no_grad() if i < first else contextlib.nullcontext()):
            h = _r_layer(h, i)
        if adapters is not None and str(i) in adapters:
            h = adapters[str(i)](h, e)
    return _rb.layernorm(h)

class _RefArmModel(nn.Module):                                        # verbatim nb13 (nhanh plain)
    def __init__(self, plain=False):
        super().__init__(); self.plain = plain; self.blocks = () if plain else tuple(_rLATE)
        self.norm = nn.LayerNorm(768); self.mld = MLDecoder(num_classes=len(LABELS),
                     initial_num_features=768, decoder_embedding=768, num_of_groups=len(LABELS))
        self.norm.load_state_dict(_sub('norm.')); self.mld.load_state_dict(_sub('mld.'))
    def forward(self, z_shared, batch=None):
        if self.plain: h = _r_forward_tail(z_shared)
        else: raise NotImplementedError('nb16 chi dung nhanh plain')
        return self.mld(self.norm(h[:, 1:]))

REF = _RefArmModel(plain=True).to(dev).eval()
for p in REF.parameters(): p.requires_grad_(False)

# --- dieu kien 3: khong dung chung module ---
_id_ref = {id(m) for m in list(REF.modules()) + list(_rb.modules())}
_id_cnd = {id(m) for m in list(_A.modules()) + list(backbone.modules())}
_shared_mod = _id_ref & _id_cnd
print(f'module dung chung REF vs CANDIDATE: {len(_shared_mod)} '
      f'{"OK (doc lap hoan toan)" if not _shared_mod else "FAIL"}')
assert not _shared_mod, 'reference va candidate dung chung module -> khong phai code path doc lap'

# ================== ba tang, qua forward hook ==================
_cap = {}
def _hook(tag): return lambda mod, inp, out: _cap.__setitem__(
    tag, (out[0] if isinstance(out, (tuple, list)) else out).detach().float())
_hs = [_rLAYERS[_rn_blocks-1].register_forward_hook(_hook('r_L1')),
       _rb.layernorm.register_forward_hook(_hook('r_L2')),
       _A.tail.blocks[-1].register_forward_hook(_hook('c_L1')),
       _A.tail.ln.register_forward_hook(_hook('c_L2'))]

_A.eval(); REF.eval()
_bat = to_dev(collate([ImgDS(IDX['val'], eval_tf)[k] for k in range(8)]))   # eval_tf: khong aug
with torch.no_grad():
    _zr = _r_forward_shared(_bat['image']);  _lr = REF(_zr, None)           # code path nb13
    _zc = forward_shared(_bat['image']);     _lc = _A(_zc)                  # code path nb16
for h in _hs: h.remove()

_L = [('L0  shared: token sau block 8',      _zr,                 _zc),
      ('L1  sau block 11 (truoc final LN)',  _cap['r_L1'],        _cap['c_L1']),
      ('L2  layernorm(h)[:, 1:] patch-only', _cap['r_L2'][:, 1:], _cap['c_L2'][:, 1:]),
      ('L3  logits sau H5 head',             _lr,                 _lc)]
print(f'\n=== IDENTITY GATE BA TANG (nguong {IDENT_TOL:.0e}) ===')
IDENT = {}
for nm, a, b in _L:
    a, b = a.float(), b.float()
    assert a.shape == b.shape, f'{nm}: shape {tuple(a.shape)} vs {tuple(b.shape)}'
    d = (a - b).abs().max().item()
    IDENT[nm.split()[0]] = d
    print(f'  {nm:<38} shape {str(tuple(a.shape)):<20} max|diff| {d:.3e}  '
          f'{"OK" if d <= IDENT_TOL else "FAIL"}')
assert all(v <= IDENT_TOL for v in IDENT.values()), (
    'identity gate FAIL — doc tang dau tien lech de dinh vi loi:\n'
    '  L1 lech -> ban sao block 9-11 hoac thu tu chay\n'
    '  chi L2 lech -> final LayerNorm hoac cho cat CLS\n'
    '  chi L3 lech -> H5 norm/mld')

# --- kiem chung nguoc: CO TINH giu CLS -> L2/L3 PHAI lech ---
with torch.no_grad():
    _bad = _A.head(_cap['c_L2'])                      # giu ca CLS = dung loi cua nb14/15
_dbad = (_bad - _lc).abs().max().item()
print(f'\n  kiem chung nguoc: neu GIU CLS thi logits lech {_dbad:.3e} '
      f'{"OK (cong nay co kha nang bat loi)" if _dbad > 1e-4 else "!! cong VO DUNG"}')
assert _dbad > 1e-4, 'giu CLS ma logits khong doi -> cong nay khong bat duoc loi gi'
print(f'  -> day chinh la loi cua nb14/15 (REVIEW_PLAN_V3_2 §3-bis). nb16 khong con.')
json.dump(dict(tol=IDENT_TOL, levels=IDENT, cls_counterfactual_diff=_dbad,
               reference='ArmModel(plain=True) verbatim tu 13_dbl_weight_screen.ipynb cell 14',
               shared_modules=0), open(f'{SC_DIR}/metrics/identity_gate.json', 'w'), indent=1)

# ================== stochasticity (§7 · 0c) ==================
cfg = backbone.config
_g = lambda k: getattr(cfg, k, None)
print(f'\n=== STOCHASTICITY ===')
for k in ('hidden_dropout_prob', 'attention_probs_dropout_prob', 'drop_path_rate',
          'layerscale_value', 'drop_path_uniform'):
    print(f'  cfg.{k:<30} {_g(k)}')
_A.train()
with torch.no_grad():
    _o1 = _A(_zc); _o2 = _A(_zc)
_A.eval()
with torch.no_grad(): _o3 = _A(_zc)
_dtt = (_o1 - _o2).abs().max().item(); _dte = (_o1 - _o3).abs().max().item()
print(f'  train() 2 lan       max|diff| {_dtt:.3e}')
print(f'  train() vs eval()   max|diff| {_dte:.3e}')
STOCHASTIC = _dtt > 1e-7
print(f'''
  KET LUAN: forward {"CO" if STOCHASTIC else "KHONG CO"} thanh phan ngau nhien.
  {"-> PHAI dung sync_rng(ep, i) tung arm tung step de dropout mask trung nhau." if STOCHASTIC else
   "-> KHONG can sync_rng. Moi khac biet giua arm chi den tu weight (do weight matrix)."}
  {"" if STOCHASTIC else "  (dropout=0 va drop_path=0 la mac dinh cua Dinov2/RAD-DINO — da xac nhan bang so.)"}''')
json.dump(dict(stochastic=bool(STOCHASTIC), train_vs_train=_dtt, train_vs_eval=_dte,
               cfg={k: _g(k) for k in ('hidden_dropout_prob','attention_probs_dropout_prob','drop_path_rate')}),
          open(f'{SC_DIR}/metrics/stochasticity.json', 'w'), indent=1)
del REF, _rb, _cap
torch.cuda.empty_cache() if dev == 'cuda' else None

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

module dung chung REF vs CANDIDATE: 0 OK (doc lap hoan toan)

=== IDENTITY GATE BA TANG (nguong 1e-06) ===
  L0  shared: token sau block 8          shape (8, 1370, 768)       max|diff| 0.000e+00  OK
  L1  sau block 11 (truoc final LN)      shape (8, 1370, 768)       max|diff| 0.000e+00  OK
  L2  layernorm(h)[:, 1:] patch-only     shape (8, 1369, 768)       max|diff| 0.000e+00  OK
  L3  logits sau H5 head                 shape (8, 13)              max|diff| 2.384e-07  OK

  kiem chung nguoc: neu GIU CLS thi logits lech 4.804e-03 OK (cong nay co kha nang bat loi)
  -> day chinh la loi cua nb14/15 (REVIEW_PLAN_V3_2 §3-bis). nb16 khong con.

=== STOCHASTICITY ===
  cfg.hidden_dropout_prob            0.0
  cfg.attention_probs_dropout_prob   0.0
  cfg.drop_path_rate                 0.0
  cfg.layerscale_value               1.0
  cfg.drop_path_uniform              None
  train() 2 lan       max|diff| 5.239e-01
  train() vs eval()   max|diff| 4.018e-01

  KET LUAN: forward CO thanh phan ngau nh

## C8 · Initial ASL logit-gradient share → `β_c` (**0b** + **0d**)

Đây là bước làm cho `R1GM` là control **thật**. Với `y = 0`, số hạng âm của ASL là

$$\ell^-(z)=-\ln(x_n)\,(1-x_n)^{\gamma_n},\qquad x_n=\min(1-\sigma(z)+0.05,\;1)$$

và độ lớn gradient tại logit là $g=\left|\partial\ell^-/\partial z\right|$. Tính bằng
**autograd trên đúng hàm loss đang dùng**, không viết lại đạo hàm bằng tay — rồi đối chiếu với
dạng đóng để chắc.

Tổng theo **ảnh** (loss tính ở mức ảnh, `y` lấy từ study):

$$G_{\text{cont},c}=\sum_{k:\text{contested}}g_{k,c},\qquad
G_{\text{ord},c}=\sum_{k:\text{blank},\,\neg\text{contested}}g_{k,c},\qquad
\beta_c=\frac{\eta G_{\text{cont},c}+G_{\text{ord},c}}{G_{\text{cont},c}+G_{\text{ord},c}}$$

`NonTarget-10` có `G_cont = 0` ⇒ `β = 1` tự động. `β_c ∈ [η, 1]` luôn — không cần clamp.

Vì sao phải **đo** thay vì suy: `(p − 0.05)^4` cực lồi. Đo trên chính hàm loss đang dùng:
`g(p=0.90) / g(p=0.10) ≈ 2.5×10^5` — **hơn năm bậc**. Nên negative gradient bị thống trị
bởi số ít cell điểm cao, và khớp theo **số ô** (V3.1) không hề khớp gradient. Đây đúng là
chỗ V3.1 sai. Cell `g` tại `p = clip = 0.05` bằng **đúng 0** — ô đó không đóng góp gì.

### Ô đóng băng — `raw == 0`

Chỉ ô **blank** được reweight. Ô `raw == 0` (known negative) giữ weight `1` ở **cả ba** arm.
Ô `raw == -1` thành **positive** qua U-Ones nên không nằm trong bài toán này.

### Chi phí

Một lượt forward toàn bộ TRAIN, **FP32, `eval()`, không augmentation** (phải xác định để `β_c`
tái lập được). Không backward qua model — autograd chỉ chạy trên hàm elementwise ở logit.

| GPU | ước |
|---|---|
| A100 40G | ~15–25 phút |
| L4 / V100 | ~40–70 phút |
| T4 | ~2–3 giờ |

Kết quả cache ra `initial_gradient_share.json`; chạy lại cell sẽ đọc cache.

In [10]:
GRAD_DIR   = f'{SC_DIR}/metrics'
LEGACY_JS  = f'{GRAD_DIR}/initial_gradient_share.json'          # lan chay truoc (eval_tf)
EVAL_JS    = f'{GRAD_DIR}/initial_gradient_share_eval.json'
TRAIN_JS   = lambda s: f'{GRAD_DIR}/initial_gradient_share_train_s{s}.json'
AUG_SEEDS  = (42, 1337)                                         # KHOA: 2 seed tang cuong
AUG_WORKERS = 4
SWEEP_BS   = 32

def asl_neg_elem(z):
    """So hang AM cua ASL (y=0) — trung cong thuc ASLoss/WeightedASL cua nb12."""
    xp = torch.sigmoid(z); xn = (1 - xp + ASL_CLIP).clamp(max=1)
    return -torch.log(xn.clamp(min=ASL_EPS)) * torch.pow(1 - xn, ASL_GN)

def asl_neg_grad(z):
    # enable_grad() BAT BUOC: C9 goi ham nay ben trong `with torch.no_grad()`.
    # Neu thieu, asl_neg_elem(z) khong dung graph du z.requires_grad=True, va
    # torch.autograd.grad bao "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        z = z.detach().float().requires_grad_(True)
        return torch.autograd.grad(asl_neg_elem(z).sum(), z)[0].abs().detach()

# ---------------- doi chieu autograd vs dang dong ----------------
_zt = torch.linspace(-8, 8, 4001, device=dev)
_ga = asl_neg_grad(_zt)
_p = torch.sigmoid(_zt); _xn = (1 - _p + ASL_CLIP).clamp(max=1); _u = (_p - ASL_CLIP).clamp(min=0)
_cf = torch.where(_p > ASL_CLIP,
                  (_u**ASL_GN/_xn - ASL_GN*torch.log(_xn)*_u**(ASL_GN-1)) * _p*(1-_p),
                  torch.zeros_like(_p))
print(f'autograd vs dang dong |g|: max|diff| {(_ga-_cf).abs().max().item():.3e}  '
      f'{"OK" if (_ga-_cf).abs().max().item() < 1e-5 else "FAIL"}')
assert (_ga - _cf).abs().max().item() < 1e-5
with torch.no_grad(): _gi = asl_neg_grad(_zt)
print(f'asl_neg_grad chay duoc TRONG torch.no_grad() va cho ket qua y het: {torch.equal(_ga, _gi)}'
      f'  (nho enable_grad())')
assert torch.equal(_ga, _gi)
_r = (asl_neg_grad(torch.tensor([math.log(.9/.1)], device=dev)).item()
      / max(asl_neg_grad(torch.tensor([math.log(.1/.9)], device=dev)).item(), 1e-30))
print(f'g(p=0.05)={asl_neg_grad(torch.tensor([math.log(.05/.95)], device=dev)).item():.1e} (=0, bi clip)'
      f' | g(0.90)/g(0.10) = {_r:,.0f}x (~{math.log10(_r):.1f} bac)')
print('  -> (p-0.05)^4 CUC LOI. Khop theo SO O (V3.1) khong khop gradient — phai do g.')

# ---------------- loader tang cuong TAI LAP DUOC ----------------
def det_loader(tf, seed, bs, workers, split='train'):
    g = torch.Generator(); g.manual_seed(int(seed))
    def _wi(wid):
        s = (int(seed)*100_003 + wid) % (2**31 - 1)
        torch.manual_seed(s); np.random.seed(s); random.seed(s)
    torch.manual_seed(int(seed))                       # cho truong hop workers=0
    return DataLoader(ImgDS(IDX[split], tf), batch_size=bs, shuffle=False, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev == 'cuda'), generator=g,
                      worker_init_fn=(_wi if workers > 0 else None), persistent_workers=False)

def _peek(seed, workers, k=3, bs=8):
    out = []
    for i, b in enumerate(det_loader(train_tf, seed, bs, workers)):
        out.append(b['image'].clone())
        if i + 1 >= k: break
    return out

print(f'\n=== KIEM TANG CUONG TAI LAP DUOC (workers={AUG_WORKERS}) ===')
for _w in (AUG_WORKERS, 0):
    _a1, _a2, _b1 = _peek(AUG_SEEDS[0], _w), _peek(AUG_SEEDS[0], _w), _peek(AUG_SEEDS[1], _w)
    _det = all(torch.equal(x, y) for x, y in zip(_a1, _a2))
    _dif = any(not torch.equal(x, y) for x, y in zip(_a1, _b1))
    print(f'  workers={_w}: cung seed -> bit-for-bit {_det} | khac seed -> anh khac {_dif}')
    if _det and _dif:
        AUG_WORKERS = _w; break
else:
    raise RuntimeError('khong tao duoc tang cuong tai lap duoc -> khong the khoa beta')
print(f'  -> dung workers={AUG_WORKERS}')

# ---------------- mot luot quet ----------------
def sweep_gradient(tf, seed, workers, tag, bs=SWEEP_BS):
    _A.eval()
    _ct = torch.from_numpy(CONTESTED.astype('float32')).to(dev)
    _ot = torch.from_numpy((IS_BLANK & ~CONTESTED).astype('float32')).to(dev)
    Gc = torch.zeros(C, dtype=torch.float64, device=dev)
    Go = torch.zeros(C, dtype=torch.float64, device=dev)
    acc, t0 = {l: {} for l in RARE3}, time.time()
    for b in tqdm(det_loader(tf, seed, bs, workers), desc=f'0b {tag}'):
        b = to_dev(b)
        with torch.no_grad(): z = _A(forward_shared(b['image']))
        g = asl_neg_grad(z); sr = b['sr']
        gc = g*_ct[sr]; Gc += gc.sum(0).double(); Go += (g*_ot[sr]).sum(0).double()
        _s = SUBJ[sr.cpu().numpy()]; _v = gc[:, RARE3_IDX].double().cpu().numpy()   # [B, 3]
        for _r, _t in zip(*np.nonzero(_v > 0)):
            _d = acc[RARE3[_t]]; _k = int(_s[_r])
            _d[_k] = _d.get(_k, 0.0) + float(_v[_r, _t])
    return dict(tag=tag, seed=int(seed), workers=int(workers), batch=int(bs),
                transform=('train_tf' if tf is train_tf else 'eval_tf'),
                seconds=time.time()-t0, n_images=int(len(IDX['train'])),
                G_cont=Gc.cpu().numpy().tolist(), G_ord=Go.cpu().numpy().tolist(),
                subj_cont_mass={l: {str(k): v for k, v in d.items()} for l, d in acc.items()})

def cached(path, fn):
    if os.path.exists(path):
        r = json.load(open(path)); print(f'  <- cache {os.path.basename(path)}'); return r
    r = fn(); json.dump(r, open(path, 'w'))
    print(f'  -> {os.path.basename(path)}  ({r["seconds"]/60:.1f} phut)'); return r

print('\n=== QUET GRADIENT ===')
if not os.path.exists(EVAL_JS) and os.path.exists(LEGACY_JS):
    shutil.copyfile(LEGACY_JS, EVAL_JS)     # SAO CHEP, khong xoa/ghi de file cu
    print(f'  migrate {os.path.basename(LEGACY_JS)} -> {os.path.basename(EVAL_JS)} (giu file cu)')
R_EVAL = cached(EVAL_JS, lambda: sweep_gradient(eval_tf, 0, 0, 'eval_tf'))
R_TR   = [cached(TRAIN_JS(s), (lambda s=s: sweep_gradient(train_tf, s, AUG_WORKERS, f'train_tf s{s}')))
          for s in AUG_SEEDS]

beta_of = lambda Gc, Go: np.where(np.asarray(Gc)+np.asarray(Go) > 0,
    (ETA*np.asarray(Gc)+np.asarray(Go))/np.maximum(np.asarray(Gc)+np.asarray(Go), 1e-30), 1.0)
GC_TR = np.sum([r['G_cont'] for r in R_TR], 0); GO_TR = np.sum([r['G_ord'] for r in R_TR], 0)
GC_EV = np.asarray(R_EVAL['G_cont']);           GO_EV = np.asarray(R_EVAL['G_ord'])
BETA_TRAIN, BETA_EVAL = beta_of(GC_TR, GO_TR), beta_of(GC_EV, GO_EV)
BETA = BETA_TRAIN                                        # <<<< PRIMARY
B_PER = [beta_of(r['G_cont'], r['G_ord']) for r in R_TR]

# ---------------- bang chinh ----------------
_nb_tr = (IS_BLANK & IS_TRAIN[:, None]).sum(0)           # mau so TRAIN-ONLY (sua loi ban dau)
GS = pd.DataFrame(dict(label=LABELS, tier=TIER, rare3=[l in RARE3 for l in LABELS],
    n_contested=[int(CONTESTED[:, j].sum()) for j in range(C)], n_blank_train=_nb_tr,
    G_cont=GC_TR, G_ord=GO_TR, cont_share=GC_TR/np.maximum(GC_TR+GO_TR, 1e-30),
    beta_train=BETA_TRAIN, beta_eval=BETA_EVAL, d_beta=BETA_TRAIN-BETA_EVAL))
GS['cell_share'] = GS.n_contested/GS.n_blank_train.clip(lower=1)
GS['enrich'] = GS.cont_share/GS.cell_share.clip(lower=1e-12)
GS.to_csv(f'{GRAD_DIR}/gradient_calibration.csv', index=False)
print('\n=== GRADIENT SHARE tai INIT — PRIMARY = train_tf pooled 2 seed ===')
print(GS.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
assert np.allclose(BETA[NONTGT_IDX], 1.0), 'NonTarget-10 phai co beta = 1'
assert ((BETA >= ETA-1e-9) & (BETA <= 1+1e-9)).all(), 'beta ngoai [eta, 1]'
print(f'\n  NonTarget-10 beta == 1 OK | beta thuoc [{BETA.min():.6f}, {BETA.max():.6f}] c [{ETA}, 1] OK')
print('  n_blank_train la mau so TRAIN-ONLY (ban dau toi dung mau so toan cohort -> enrich bi thoi ~1.5x)')

# ---------------- train vs eval, va do nhay theo seed tang cuong ----------------
_r3 = GS[GS.rare3]
print('\n=== beta_train vs beta_eval, va NHIEU giua 2 augmentation seed ===')
_cmp = pd.DataFrame(dict(label=[LABELS[j] for j in RARE3_IDX],
    beta_s42=[B_PER[0][j] for j in RARE3_IDX], beta_s1337=[B_PER[1][j] for j in RARE3_IDX],
    beta_train=[BETA_TRAIN[j] for j in RARE3_IDX], beta_eval=[BETA_EVAL[j] for j in RARE3_IDX]))
_cmp['seed_spread'] = (_cmp.beta_s42-_cmp.beta_s1337).abs()
_cmp['train_vs_eval'] = (_cmp.beta_train-_cmp.beta_eval).abs()
print(_cmp.to_string(index=False, float_format=lambda v: f'{v:.6f}'))
_ss, _te = float(_cmp.seed_spread.max()), float(_cmp.train_vs_eval.max())
print(f'\n  max |beta(s42) - beta(s1337)|   = {_ss:.6f}   <- nhieu noi tai cua tang cuong')
print(f'  max |beta_train - beta_eval|    = {_te:.6f}   <- hieu ung he thong cua tang cuong')
print(f'''  {"train_vs_eval > seed_spread -> tang cuong CO tac dong he thong, dung beta_train la dung."
        if _te > _ss else
        "train_vs_eval <= seed_spread -> hieu ung eval-vs-train KHONG tach duoc khoi nhieu tang cuong;"}
  {"" if _te > _ss else "  beta_train van la primary (dung phan phoi training that), nhung tranh luan nay la moot."}''')

print('\n=== cell share vs GRADIENT share (cho V3.1 sai) — mau so TRAIN-ONLY ===')
print(_r3[['label','n_contested','n_blank_train','cell_share','cont_share','enrich']]
      .to_string(index=False, float_format=lambda v: f'{v:.6f}'))
print(f'  enrich != 1.00 -> khop theo so o KHONG khop gradient.')

# ---------------- co can thiep ----------------
_tot_blank = float((GC_TR+GO_TR).sum()); _r3m = float((GC_TR+GO_TR)[RARE3_IDX].sum())
_removed = float((1-ETA)*GC_TR[RARE3_IDX].sum())
print(f'''
=== CO CAN THIEP (doc truoc khi cam ket ~40 gio GPU) ===
  tong negative-gradient mass cua o BLANK, 13 nhan : {_tot_blank:12,.0f}
  phan cua Rare-3                                   : {_r3m:12,.0f}  ({_r3m/_tot_blank:.2%})
  R2 xoa (1-eta)*G_cont cua Rare-3                  : {_removed:12,.0f}  ({_removed/_tot_blank:.2%} tong)
  -> toan bo thi nghiem dich {_removed/_tot_blank:.2%} khoi luong gradient am.
     Dat canh REVIEW_LABEL_NOISE_24H: nhieu nhan dich Rare-3 mAP +/-0.05, cong dang xet +0.002.
     Day KHONG phai ly do dung; day la co hieu ung ky vong, va no lam kich ban
     "D1 ~ 0 & D2 ~ 0" thanh ket cuc xac suat cao nhat — dung nhu Q4 da du lieu truoc.''')

# ---------------- concentration gate ----------------
def norm_subj(r):
    """-> {label: {subject: mass}}. File legacy (eval) chi co dang gop -> '_pooled'."""
    m = r['subj_cont_mass']
    return m if (m and isinstance(next(iter(m.values())), dict)) else {'_pooled': m}

def subj_series(rs, label=None):
    d = {}
    for r in rs:
        m = norm_subj(r)
        srcs = ([m[label]] if label in m else []) if label is not None else list(m.values())
        for src in srcs:
            for k, v in src.items(): d[k] = d.get(k, 0.0) + v
    return pd.Series(d, dtype='float64').sort_values(ascending=False)

def top1(sc):
    if not len(sc): return 0.0, 0, 0
    k = max(1, int(np.ceil(0.01*len(sc))))
    return float(sc.iloc[:k].sum()/max(float(sc.sum()), 1e-30)), k, len(sc)

SUBJ_CONT = subj_series(R_TR)                                  # gop Rare-3, train_tf = PRIMARY
_t_tr, _k_tr, _n_tr = top1(SUBJ_CONT)
_t_ev, _k_ev, _n_ev = top1(subj_series([R_EVAL]))
print()
print(f'=== CONCENTRATION GATE — theo GRADIENT MASS (§4.4) ===')
print(f'  PRIMARY  train_tf gop Rare-3 : {_n_tr:,} subject | top 1% = {_k_tr} -> {_t_tr:.4%} '
      f'(nguong <= {GATE_TOP1PCT_SHARE:.0%})  {"OK" if _t_tr <= GATE_TOP1PCT_SHARE else "FAIL"}')
print(f'  sensitivity eval_tf          : {_n_ev:,} subject | top 1% = {_k_ev} -> {_t_ev:.4%}  '
      f'{"OK" if _t_ev <= GATE_TOP1PCT_SHARE else "OK/FAIL (sensitivity)"}')
_PC = []
for l in RARE3:
    sc = subj_series(R_TR, l); t_, k_, n_ = top1(sc)
    _PC.append(dict(label=l, subjects=n_, top1pct_k=k_, top1pct_share=t_,
                    over_threshold=bool(t_ > GATE_TOP1PCT_SHARE)))
PC = pd.DataFrame(_PC); PC.to_csv(f'{GRAD_DIR}/concentration_by_class.csv', index=False)
print()
print('  --- DIAGNOSTIC: concentration TUNG NHAN (gate khoa truoc la so GOP, khong phai so nay) ---')
print(PC.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
if PC.over_threshold.any():
    print(f'  ! {list(PC[PC.over_threshold].label)} vuot {GATE_TOP1PCT_SHARE:.0%} khi xet RIENG.')
    print('    Gate khoa truoc (§4.4) la mot nguong duy nhat -> doc tu nhien la so GOP, va so gop PASS.')
    print('    KHONG tu dong fail o day: dat them tieu chi sau khi thay so la doi gate hau nghiem.')
    print('    Nhung phai GHI vao bao cao: nhan do co bang chung tap trung o it benh nhan hon.')
else:
    print('  khong nhan nao vuot nguong khi xet rieng.')
_part = json.load(open(f'{SC_DIR}/metrics/sufficiency_partial.json'))
_ok = _part['gate_cells_subjects_pass'] and _t_tr <= GATE_TOP1PCT_SHARE
json.dump(dict(**_part, primary='train_tf pooled seeds ' + str(list(AUG_SEEDS)),
               top1pct_gradient_share_train=_t_tr, top1pct_gradient_share_eval=_t_ev,
               gate_pass=bool(_ok), eta=ETA, aug_workers=int(AUG_WORKERS),
               beta_train={l: float(b) for l, b in zip(LABELS, BETA_TRAIN)},
               beta_eval={l: float(b) for l, b in zip(LABELS, BETA_EVAL)},
               beta_per_aug_seed={str(s): {l: float(b) for l, b in zip(LABELS, bp)}
                                  for s, bp in zip(AUG_SEEDS, B_PER)},
               max_seed_spread=_ss, max_train_vs_eval=_te,
               intervention_share_of_blank_negative_mass=_removed/_tot_blank),
          open(f'{GRAD_DIR}/gradient_calibration.json', 'w'), indent=1)
print(f'''
=== GO / NO-GO cua 0a + 0b: {"GO" if _ok else "NO-GO"} ===
  {"Du dieu kien de R2 la primary. Sang 0c (C9)." if _ok else
   "R2 MAT primary status (§4.4). KHONG duoc sua manifest roi tiep tuc claim cung estimand."}
  beta PRIMARY = beta_train (pooled 2 aug seed). beta_eval giu lam sensitivity.''')

autograd vs dang dong |g|: max|diff| 5.960e-08  OK
asl_neg_grad chay duoc TRONG torch.no_grad() va cho ket qua y het: True  (nho enable_grad())
g(p=0.05)=0.0e+00 (=0, bi clip) | g(0.90)/g(0.10) = 252,604x (~5.4 bac)
  -> (p-0.05)^4 CUC LOI. Khop theo SO O (V3.1) khong khop gradient — phai do g.

=== KIEM TANG CUONG TAI LAP DUOC (workers=4) ===
  workers=4: cung seed -> bit-for-bit True | khac seed -> anh khac True
  -> dung workers=4

=== QUET GRADIENT ===
  migrate initial_gradient_share.json -> initial_gradient_share_eval.json (giu file cu)
  <- cache initial_gradient_share_eval.json


0b train_tf s42:   0%|          | 0/7579 [00:00<?, ?it/s]

  -> initial_gradient_share_train_s42.json  (27.4 phut)


0b train_tf s1337:   0%|          | 0/7579 [00:00<?, ?it/s]

  -> initial_gradient_share_train_s1337.json  (27.8 phut)

=== GRADIENT SHARE tai INIT — PRIMARY = train_tf pooled 2 seed ===
                     label   tier  rare3  n_contested  n_blank_train      G_cont        G_ord  cont_share  beta_train  beta_eval   d_beta  cell_share   enrich
               Atelectasis   head  False            0         109822    0.000000 25730.584831    0.000000    1.000000   1.000000 0.000000    0.000000 0.000000
              Cardiomegaly   head  False            0         104474    0.000000 29308.037494    0.000000    1.000000   1.000000 0.000000    0.000000 0.000000
             Consolidation medium  False            0         132668    0.000000 18032.250327    0.000000    1.000000   1.000000 0.000000    0.000000 0.000000
                     Edema   head  False            0         104976    0.000000 18617.497804    0.000000    1.000000   1.000000 0.000000    0.000000 0.000000
Enlarged_Cardiomediastinum medium  False            0         133619    0.00000

## C9 · Weight matrix + smoke 200 step (**0c** + **0d**)

`W[arm]` là `[N_study, 13]`; mỗi ảnh thừa hưởng weight của study nó thuộc về.

```
R0     : 1 khap noi
R1GM   : beta_c cho MOI o blank (contested + ordinary), 1 cho o khac
R2     : eta cho o CONTESTED blank, 1 cho o khac
```

Loss chuẩn hoá bằng **hằng số** `B_actual × C`:

$$L=\frac{1}{B\cdot C}\sum_{k,c}w_{k,c}\,\ell_{k,c}$$

Với `w ≡ 1` đây **đúng bằng** `ASLoss(...).mean()` của nb11–15 ⇒ hằng số chuẩn hoá không đổi,
`lr = 1e-5` vẫn ở đúng thang cũ. **Không** chia `Σw`: nếu chia thì 10 nhãn non-target sẽ nhận
gradient khác nhau giữa các arm chỉ vì mẫu số khác — làm hỏng chính cái mà control cần giữ.

Smoke 200 step kiểm **hiện thực và chi phí**, không kiểm hiệu năng:

| kiểm | tiêu chí |
|---|---|
| blocks 0–8 | **bất biến bit-for-bit** — hard failure nếu đổi |
| blocks 9–11 + `backbone.layernorm` | **có** update — hard failure nếu không |
| H5 `norm`/`mld` ở ep1 | **bất biến** (head warmup) |
| loss, grad norm | hữu hạn — hard failure nếu NaN/Inf |
| gradient-mass ratio `R1GM/R2` | ≈ 1 tại step 0 → xác nhận khớp thực nghiệm |
| drift, cosine, VRAM, s/step | **DIAGNOSTIC** — không dùng để dừng run (§7) |

Ngoại suy `√t` từ 200 step **không** dùng để đặt ngưỡng. Plan §7 bỏ drift gate vì hai lý do:
drift không đơn điệu với model validity, và ngoại suy `√t` sai khi có warmup + clipping.

In [12]:
# ---------------- weight matrix (dung BETA = beta_train) ----------------
def build_W(arm):
    W = np.ones((N, C), 'float32')
    if arm == 'R0':   return W
    if arm == 'R1GM': W[IS_BLANK] = np.broadcast_to(BETA.astype('float32'), (N, C))[IS_BLANK]
    if arm == 'R2':   W[CONTESTED & IS_BLANK] = ETA
    return W
W  = {a: build_W(a) for a in ARMS}
Wt = {a: torch.from_numpy(W[a]).to(dev) for a in ARMS}
BLANK_T = torch.from_numpy(IS_BLANK.astype('float32')).to(dev)
print('=== weight matrix (beta = beta_train) ===')
for a in ARMS:
    w = W[a]
    print(f'  {a:<5} uniq {np.unique(np.round(w,6))[:6]}... | mean {w.mean():.6f} | '
          f'n(w!=1) {int((w!=1).sum()):>9,} | n(w=={ETA}) {int(np.isclose(w,ETA).sum()):>8,}')
assert (W['R0'] == 1).all()
assert (W['R2'][~(CONTESTED & IS_BLANK)] == 1).all() and np.isclose(W['R2'][CONTESTED & IS_BLANK], ETA).all()
assert (W['R1GM'][~IS_BLANK] == 1).all()
assert np.allclose(W['R1GM'][:, NONTGT_IDX], 1.0), 'NonTarget-10 bi doi weight o R1GM -> control sai'
print(f'  NonTarget-10 weight == 1 o CA BA arm: '
      f'{all(np.allclose(W[a][:, NONTGT_IDX], 1.0) for a in ARMS)}')
print('  -> can thiep chi cham 3 nhan Rare-3; 10 nhan con lai la placebo channel.')

class CellWASL(nn.Module):
    """ASL + weight tung O. w==1 => trung ASLoss().mean() cua nb11-15 (chuan hoa HANG SO B*C)."""
    def __init__(s): super().__init__(); s.gn, s.gp, s.clip, s.eps = ASL_GN, ASL_GP, ASL_CLIP, ASL_EPS
    def forward(s, x, y, w, parts=False):
        y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        Lp, Ln = -lp*mod, -ln*mod
        loss = (w*(Lp+Ln)).sum()/(x.shape[0]*x.shape[1])
        if parts: return loss, (w*Lp).mean(0).detach(), (w*Ln).mean(0).detach()
        return loss
CRIT = CellWASL().to(dev)

class _ASLref(nn.Module):
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = (1-xp+ASL_CLIP).clamp(max=1)
        los = y*torch.log(xp.clamp(min=ASL_EPS)) + (1-y)*torch.log(xn.clamp(min=ASL_EPS))
        pt = xp*y + xn*(1-y); g = ASL_GP*y + ASL_GN*(1-y)
        return -(los*torch.pow(1-pt, g)).mean()
torch.manual_seed(0)
_x = torch.randn(64, C, device=dev); _y = (torch.rand(64, C, device=dev) < .15).float()
_rf = _ASLref()(_x, _y); _nw = CRIT(_x, _y, torch.ones_like(_x))
print(f'\nCellWASL(w=1) == ASLoss cua nb11-15: {_rf.item():.10f} vs {_nw.item():.10f} '
      f'| diff {abs(_rf.item()-_nw.item()):.3e} {"OK" if torch.equal(_rf,_nw) else "FAIL"}')
assert torch.equal(_rf, _nw), 'w=1 khong tai tao ASL -> hang so chuan hoa da doi, lr sai thang'

# ---------------- optimizer / sched / clip ----------------
def make_opt(m):
    def sp(ps, lr):
        d = [p for p in ps if p.ndim > 1]; nd = [p for p in ps if p.ndim <= 1]
        return [{'params': d, 'lr': lr, 'weight_decay': WD},
                {'params': nd, 'lr': lr, 'weight_decay': 0.0}]
    return torch.optim.AdamW(sp(tail_params(m), LR_BLOCKS) + sp(head_params(m), LR_HEAD))

def make_sched(opt, total, warm):
    cw = lambda s: s/max(warm,1) if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    return torch.optim.lr_scheduler.LambdaLR(opt, [cw]*len(opt.param_groups))

set_head_trainable = lambda m, on: [p.requires_grad_(on) for p in head_params(m)]

def clip_arm(m):
    t = [p for p in tail_params(m) if p.grad is not None]
    h = [p for p in head_params(m) if p.grad is not None]
    return (float(nn.utils.clip_grad_norm_(t, CLIP_BLOCKS)) if t else 0.0,
            float(nn.utils.clip_grad_norm_(h, CLIP_HEAD)) if h else 0.0)

def sync_rng(ep, i):
    s = (SEED*1_000_003 + ep*100_003 + i) % (2**31-1)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---------------- probe: THROUGHPUT + VRAM ----------------
def probe_batch(cands=(32, 64, 128, 256), warm=2, meas=5):
    if dev != 'cuda': return 2, pd.DataFrame()
    A = [build_arm() for _ in ARMS]
    for m in A: set_head_trainable(m, False); m.train()      # dung policy ep1
    tot = torch.cuda.get_device_properties(0).total_memory/1e9
    rows = []
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            bb = to_dev(collate([ImgDS(IDX['train'], train_tf)[k] for k in range(B)]))
            for it in range(warm+meas):
                if it == warm: torch.cuda.synchronize(); t0 = time.time()
                z = forward_shared(bb['image'])
                for m in A:
                    CRIT(m(z), bb['y'], Wt['R0'][bb['sr']]).backward(); m.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            dt = (time.time()-t0)/meas; pk = torch.cuda.max_memory_allocated()/1e9
            rows.append(dict(batch=B, s_per_step=dt, img_per_s=B/dt, peak_gb=pk,
                             frac=pk/tot, ok=bool(pk/tot < 0.80)))
            print(f'  B={B:4d}  {dt:6.3f} s/step  {B/dt:7.1f} img/s  peak {pk:5.1f}/{tot:.0f} GB '
                  f'({pk/tot:5.1%})  {"OK" if pk/tot < 0.80 else "QUA 80% -> dung"}')
            del bb
            if pk/tot >= 0.80: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B:4d}  OOM'); torch.cuda.empty_cache(); break
    del A; torch.cuda.empty_cache()
    PB = pd.DataFrame(rows); _ok = PB[PB.ok]
    return (int(_ok.loc[_ok.img_per_s.idxmax(), 'batch']) if len(_ok) else cands[0]), PB

print(f'\n=== PROBE BATCH ({len(ARMS)} arm x 3 block trainable x {N_PATCH+1} token) ===')
BATCH, PB = probe_batch()
ACCUM = max(1, EFF//BATCH)
if len(PB):
    _best = PB.loc[PB.batch == BATCH].iloc[0]
    print(f'-> chon BATCH {BATCH} theo throughput ({_best.img_per_s:.1f} img/s, '
          f'{_best.frac:.1%} VRAM) | accum {ACCUM} -> effective {BATCH*ACCUM} (muc tieu {EFF})')
    print(f'   nhanh hon B=32: {_best.img_per_s/PB.iloc[0].img_per_s:.2f}x')
PB.to_csv(f'{SC_DIR}/metrics/probe_batch.csv', index=False)
assert BATCH*ACCUM == EFF or BATCH > EFF, f'BATCH*ACCUM={BATCH*ACCUM} != EFF={EFF}'

# ---------------- smoke 200 step ----------------
SMOKE_STEPS = 200
def smoke(steps=SMOKE_STEPS, batch=BATCH):
    base = build_arm(SEED); ST = {}
    for a in ARMS:
        m = Arm().to(dev); m.load_state_dict({k: v.detach().clone() for k, v in base.state_dict().items()})
        o = make_opt(m)
        ST[a] = dict(m=m, opt=o, sch=make_sched(o, steps//ACCUM+1, max(1, int(WARMUP_FRAC*steps//ACCUM))))
    _sd = [ST[a]['m'].state_dict() for a in ARMS]
    assert all(torch.equal(x, y) for d in _sd[1:] for x, y in zip(_sd[0].values(), d.values())), \
        'cac arm KHONG cung init -> khong con la paired'
    print(f'{len(ARMS)} arm cung init bit-for-bit')
    FROZEN0 = {k: v.detach().clone() for k, v in backbone.state_dict().items()}
    INIT = {a: {k: v.detach().clone() for k, v in ST[a]['m'].state_dict().items()} for a in ARMS}
    for a in ARMS: set_head_trainable(ST[a]['m'], False); ST[a]['m'].train()   # ep1: head frozen
    print(f'ep1 policy: head frozen | tail trainable ({_np(tail_params(ST[ARMS[0]]["m"])):,} params)')

    dl = loader('train', train_tf, True, batch, workers=4)
    LOGS = {a: [] for a in ARMS}; GM = []
    torch.manual_seed(SEED*7919 + 0)
    it = iter(dl); t0 = time.time()
    if dev == 'cuda': torch.cuda.reset_peak_memory_stats()
    for i in tqdm(range(steps), desc='0c smoke'):
        try: b = next(it)
        except StopIteration: it = iter(dl); b = next(it)
        b = to_dev(b); z = forward_shared(b['image'])
        step_now = ((i+1) % ACCUM == 0); gm = {}
        for a in ARMS:
            e = ST[a]
            if STOCHASTIC: sync_rng(0, i)
            zz = e['m'](z)
            loss = CRIT(zz, b['y'], Wt[a][b['sr']])
            assert torch.isfinite(loss), f'HARD FAILURE: loss khong huu han o arm {a} step {i}'
            (loss/ACCUM).backward()
            _g = asl_neg_grad(zz)*(1-b['y'])*Wt[a][b['sr']]
            gm[a] = float(_g[:, RARE3_IDX].sum())
            gm[a+'_blank'] = float((_g*BLANK_T[b['sr']])[:, RARE3_IDX].sum())
            if step_now:
                nt, nh = clip_arm(e['m'])
                assert math.isfinite(nt) and math.isfinite(nh), f'HARD FAILURE: grad norm {a} step {i}'
                e['opt'].step(); e['sch'].step(); e['opt'].zero_grad(set_to_none=True)
                LOGS[a].append(dict(step=i+1, loss=float(loss.detach()), gn_tail=nt, gn_head=nh,
                                    lr=e['opt'].param_groups[0]['lr']))
        GM.append(gm)
    el = time.time() - t0

    # ---- HARD FAILURE (danh sach duong 6 muc) ----
    _fn = lambda ps: float(torch.linalg.vector_norm(
        torch.stack([torch.linalg.vector_norm(p.detach().float()) for p in ps])))
    print(f'\n=== HARD FAILURE CHECKS (danh sach duong, §7 · REVIEW §2·Q3) ===')
    _bad = {k: (FROZEN0[k]-v).abs().max().item() for k, v in backbone.state_dict().items()
            if not torch.equal(FROZEN0[k], v)}
    print(f'  1. blocks 0-8 bat bien bit-for-bit         {"OK" if not _bad else f"FAIL {list(_bad)[:3]}"}')
    assert not _bad, f'HARD FAILURE: shared frozen backbone bi doi: {list(_bad)[:5]}'
    R = []
    for a in ARMS:
        sd = ST[a]['m'].state_dict()
        dt = max((sd[k]-INIT[a][k]).abs().max().item() for k in sd if k.startswith('tail.'))
        dh = max((sd[k]-INIT[a][k]).abs().max().item() for k in sd if k.startswith('head.'))
        dln = max((sd[k]-INIT[a][k]).abs().max().item() for k in sd if k.startswith('tail.ln.'))
        rel = float(torch.linalg.vector_norm(torch.stack([(sd[k]-INIT[a][k]).float().flatten().norm()
                    for k in sd if k.startswith('tail.')]))) / max(_fn([INIT[a][k] for k in sd
                    if k.startswith('tail.')]), 1e-12)
        cs = [float(F.cosine_similarity(sd[k].float().flatten(), INIT[a][k].float().flatten(), 0))
              for k in sd if k.startswith('tail.') and sd[k].ndim > 1]
        R.append(dict(arm=a, d_tail=dt, d_tail_ln=dln, d_head=dh, rel_drift_tail=rel,
                      min_cos_tail=float(np.min(cs)), final_loss=LOGS[a][-1]['loss'],
                      gn_tail=float(np.mean([r['gn_tail'] for r in LOGS[a][-20:]])),
                      gn_head=float(np.mean([r['gn_head'] for r in LOGS[a][-20:]]))))
    RD = pd.DataFrame(R)
    for n_, c_, ok_ in ((2, 'blocks 9-11 CO update (d_tail > 0)', (RD.d_tail > 0).all()),
                        (3, 'backbone.layernorm CO update', (RD.d_tail_ln > 0).all()),
                        (4, 'H5 head bat bien o ep1 (d_head == 0)', (RD.d_head == 0).all()),
                        (5, 'loss & grad norm huu han', True),
                        (6, 'output khong thanh hang so', (RD.min_cos_tail > 0.9).all())):
        print(f'  {n_}. {c_:<41} {"OK" if ok_ else "FAIL"}')
    assert (RD.d_tail > 0).all() and (RD.d_tail_ln > 0).all(), 'HARD FAILURE: tail khong update'
    assert (RD.d_head == 0).all(), 'HARD FAILURE: head bi update trong head-warmup epoch'
    print(f'\n=== DIAGNOSTIC (KHONG dung de dung run / loai checkpoint — §7) ===')
    print(RD.to_string(index=False, float_format=lambda v: f'{v:.6g}'))

    # ---- 0d: khop gradient, POOLED + BOOTSTRAP CI ----
    _rng = np.random.default_rng(SEED); _ix = _rng.integers(0, len(GM), (N_BOOT, len(GM)))
    def _pooled(k1, k2):
        a = np.array([g[k1] for g in GM]); b = np.array([g[k2] for g in GM])
        pt = float(a.sum()/max(b.sum(), 1e-30))
        bo = a[_ix].sum(1)/np.maximum(b[_ix].sum(1), 1e-30)
        return pt, float(np.percentile(bo, 2.5)), float(np.percentile(bo, 97.5))
    print(f'\n=== 0d · KHOP GRADIENT — POOLED {len(GM)} batch + bootstrap CI ({N_BOOT}) ===')
    CI = {}
    for nm, k1, k2 in (('blank-only  (phep thu SAC — dung tap ma beta khop)', 'R1GM_blank', 'R2_blank'),
                       ('all-negative (gom o raw==0, w=1 moi arm -> pha loang)', 'R1GM', 'R2')):
        pt, lo, hi = _pooled(k1, k2); CI[k1] = dict(ratio=pt, lo=lo, hi=hi)
        print(f'  R1GM/R2 {nm:<52} {pt:.6f}  CI95 [{lo:.6f}, {hi:.6f}]  '
              f'{"CHUA 1.0 -> khop" if lo <= 1.0 <= hi else "KHONG chua 1.0"}')
    _s0 = GM[0]['R1GM_blank']/max(GM[0]['R2_blank'], 1e-30)
    print(f'  sanity mot batch (step 0): {_s0:.6f}  <- CHI la sanity check.')
    print(f'    ~{batch} anh/batch => rat it o contested, ma g trai >5 bac => phuong sai lon.')
    print(f'    So duoc CLAIM la pooled + CI o tren, KHONG phai so nay.')
    _m = np.array([[g['R0'], g['R1GM'], g['R2']] for g in GM]).mean(0)
    print(f'  mass trung binh (all-neg): R0 {_m[0]:.4f} | R1GM {_m[1]:.4f} | R2 {_m[2]:.4f}'
          f'  | R1GM/R0 {_m[1]/_m[0]:.4f} | R2/R0 {_m[2]/_m[0]:.4f}')

    # ---- runtime: train + VAL ----
    _spe = math.ceil(len(IDX['train'])/batch)
    h_tr = _spe*el/steps/3600
    for a in ARMS: ST[a]['m'].eval()
    torch.cuda.synchronize() if dev == 'cuda' else None
    tv = time.time(); nv = 0
    with torch.no_grad():
        for b in tqdm(loader('val', eval_tf, False, batch, workers=4), desc='val pass', leave=False):
            b = to_dev(b); z = forward_shared(b['image'])
            for a in ARMS: torch.sigmoid(ST[a]['m'](z))
            nv += b['image'].shape[0]
    torch.cuda.synchronize() if dev == 'cuda' else None
    h_val = (time.time()-tv)/3600
    print(f'\n=== RUNTIME (do that, gom ca VAL) ===')
    print(f'  smoke {steps} step x {len(ARMS)} arm @B={batch} | {el:.0f}s | {el/steps:.3f} s/step '
          f'| {batch/(el/steps):.1f} img/s')
    if dev == 'cuda':
        print(f'  peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} / '
              f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
    print(f'  val pass  {nv:,} anh x {len(ARMS)} arm | {h_val*3600:.0f}s')
    print(f'  1 epoch = train {h_tr:.2f} h + val {h_val:.2f} h = {h_tr+h_val:.2f} h '
          f'({_spe:,} step/epoch)')
    print(f'  {EPOCHS} epoch -> {EPOCHS*(h_tr+h_val):.1f} h/seed  ->  3 seed = '
          f'{3*EPOCHS*(h_tr+h_val):.0f} h  <<< CON SO CAM KET')
    _snap = _np(tail_params(ST[ARMS[0]]['m']) + head_params(ST[ARMS[0]]['m']))*4/1e9
    print(f'  snapshot {_snap*len(ARMS):.2f} GB/epoch ({len(ARMS)} arm) -> '
          f'{_snap*len(ARMS)*EPOCHS:.1f} GB/seed -> {_snap*len(ARMS)*EPOCHS*3:.0f} GB (3 seed)')
    print(f'  resume (params + 2 moment AdamW) {_snap*len(ARMS)*3:.2f} GB, ghi de moi epoch')
    print(f'  -> {EPOCHS*(h_tr+h_val):.1f} h/seed VUOT XA mot session Colab: '
          f'resume theo epoch la BAT BUOC trong train loop.')
    json.dump(dict(steps=steps, batch=batch, accum=ACCUM, seconds=el, s_per_step=el/steps,
                   img_per_s=batch/(el/steps), steps_per_epoch=_spe,
                   hours_train_per_epoch=h_tr, hours_val_per_epoch=h_val,
                   hours_per_seed=EPOCHS*(h_tr+h_val), hours_3seed=3*EPOCHS*(h_tr+h_val),
                   n_val_images=int(nv), snapshot_gb_per_epoch=_snap*len(ARMS),
                   peak_vram_gb=(torch.cuda.max_memory_allocated()/1e9 if dev == 'cuda' else None),
                   grad_match=CI, sanity_single_batch=_s0, n_boot=N_BOOT,
                   beta_source='train_tf pooled aug seeds ' + str(list(AUG_SEEDS)),
                   gm_per_batch=GM, diagnostics=RD.to_dict('records'),
                   note='drift/cosine la DIAGNOSTIC ONLY (§7). Claim = pooled ratio + CI, khong phai step 0.'),
              open(f'{SC_DIR}/metrics/smoke_200step.json', 'w'), indent=1)
    del ST, base, INIT, FROZEN0
    torch.cuda.empty_cache() if dev == 'cuda' else None
    return RD

SMOKE = smoke()

=== weight matrix (beta = beta_train) ===
  R0    uniq [1.]... | mean 1.000000 | n(w!=1)         0 | n(w==0.1)        0
  R1GM  uniq [0.859643 0.861313 0.932887 1.      ]... | mean 0.974023 | n(w!=1)   617,798 | n(w==0.1)        0
  R2    uniq [0.1 1. ]... | mean 0.991724 | n(w!=1)    25,238 | n(w==0.1)   25,238
  NonTarget-10 weight == 1 o CA BA arm: True
  -> can thiep chi cham 3 nhan Rare-3; 10 nhan con lai la placebo channel.

CellWASL(w=1) == ASLoss cua nb11-15: 0.1613069624 vs 0.1613069624 | diff 0.000e+00 OK

=== PROBE BATCH (3 arm x 3 block trainable x 1370 token) ===
  B=  32   0.690 s/step     46.4 img/s  peak   9.9/102 GB ( 9.7%)  OK
  B=  64   1.339 s/step     47.8 img/s  peak  18.4/102 GB (18.0%)  OK
  B= 128   2.669 s/step     48.0 img/s  peak  35.3/102 GB (34.6%)  OK
  B= 256   5.390 s/step     47.5 img/s  peak  69.2/102 GB (67.8%)  OK
-> chon BATCH 128 theo throughput (48.0 img/s, 34.6% VRAM) | accum 2 -> effective 256 (muc tieu 256)
   nhanh hon B=32: 1.03x
3 arm cung 

0c smoke:   0%|          | 0/200 [00:00<?, ?it/s]


=== HARD FAILURE CHECKS (danh sach duong, §7 · REVIEW §2·Q3) ===
  1. blocks 0-8 bat bien bit-for-bit         OK
  2. blocks 9-11 CO update (d_tail > 0)        OK
  3. backbone.layernorm CO update              OK
  4. H5 head bat bien o ep1 (d_head == 0)      OK
  5. loss & grad norm huu han                  OK
  6. output khong thanh hang so                OK

=== DIAGNOSTIC (KHONG dung de dung run / loai checkpoint — §7) ===
 arm      d_tail   d_tail_ln  d_head  rel_drift_tail  min_cos_tail  final_loss   gn_tail  gn_head
  R0  0.00037121 0.000341341       0      0.00073959      0.999999   0.0442364 0.0181242        0
R1GM 0.000366278 0.000334695       0       0.0007351      0.999999   0.0441333 0.0180824        0
  R2 0.000365155 0.000337824       0     0.000737297      0.999999   0.0441848 0.0181235        0

=== 0d · KHOP GRADIENT — POOLED 200 batch + bootstrap CI (2000) ===
  R1GM/R2 blank-only  (phep thu SAC — dung tap ma beta khop)   1.000294  CI95 [0.993445, 1.007608]  CHUA 1.

val pass:   0%|          | 0/406 [00:00<?, ?it/s]


=== RUNTIME (do that, gom ca VAL) ===
  smoke 200 step x 3 arm @B=128 | 540s | 2.701 s/step | 47.4 img/s
  peak VRAM 36.9 / 102 GB
  val pass  51,880 anh x 3 arm | 528s
  1 epoch = train 1.42 h + val 0.15 h = 1.57 h (1,895 step/epoch)
  8 epoch -> 12.5 h/seed  ->  3 seed = 38 h  <<< CON SO CAM KET
  snapshot 0.33 GB/epoch (3 arm) -> 2.6 GB/seed -> 8 GB (3 seed)
  resume (params + 2 moment AdamW) 0.99 GB, ghi de moi epoch
  -> 12.5 h/seed VUOT XA mot session Colab: resume theo epoch la BAT BUOC trong train loop.


---

# ⛔ DỪNG — hết phần tiền-GPU

Notebook này **không** train. Bốn bước của plan §13 đã xong; đọc số rồi mới quyết.

## Đọc theo thứ tự này

| # | file | quyết định nó chi phối |
|---|---|---|
| 1 | `metrics/sufficiency_gate.json` | **`gate_pass`** — nếu `false` thì `R2` mất primary status (§4.4) |
| 2 | `metrics/temporal_manifest_funnel.csv` | bước nào ăn hết cell: cửa sổ 7d, same-view, hay timestamp |
| 3 | `metrics/contested_cells_by_class.csv` | `past` vs `future` của `Fracture` — quyết định claim được phép nói gì |
| 4 | `metrics/initial_gradient_share.csv` | `β_c` và `cont_share`; đối chiếu `cell_share` |
| 5 | `metrics/smoke_200step.json` | `hours_per_seed × 3` — chi phí cam kết trước |
| 6 | `metrics/timestamp_audit.json` | rule cho `00:00:00` **chưa chốt** (§12) |
| 7 | `metrics/identity_gate.json` | ba tầng phải `≤ 1e-6` |

## Ba đường ra

**A · `gate_pass = true`** → khoá manifest hash + `β_c` + commit, rồi viết train loop. Cam kết
`3 seed × 8 epoch × 3 arm` **trước**, chạy cả ba seed **bất kể** dấu delta của seed 42 (§11 —
đây chính là lỗi selection bias mà V3.2 §13 mắc phải).

**B · fail vì cell/subject** → `R2` mất primary status. **Không** sửa manifest rồi tiếp tục
claim cùng estimand. Được chạy `cap ≤ 3 cell/subject` như sensitivity nhưng phải ghi
exploratory (§4.4).

**C · fail vì concentration** → như B. Nếu top 1% subject chiếm quá `15%` contested gradient
mass thì `R2` thực chất đang học một nhóm nhỏ bệnh nhân, không phải một hiện tượng nhãn.

## Việc cần làm trước khi viết train loop — không tự quyết

1. **Rule `00:00:00`** (§12) — hiện `TS_OK` chỉ loại `NaT`. Đọc `timestamp_audit.json` rồi chốt.
2. **Chi phí.** Blocks 9–11 trainable × 1,370 token × 3 arm sẽ đắt hơn nb15 nhiều. Con số thật ở
   `smoke_200step.json`; nếu `hours_per_seed × 3` vượt hạn mức Colab thì phải quyết trước:
   giảm `BATCH` (chậm hơn), hay chạy tuần tự từng arm (mất tính paired trên cùng batch — **đổi
   thiết kế**, không phải đổi tham số).
3. **Nếu `STOCHASTIC = False`** thì `sync_rng` là no-op vô hại; nhưng ghi vào prereg rằng khác
   biệt giữa arm hoàn toàn tất định theo weight matrix. Đó là điều kiện paired **mạnh hơn**
   nb12–15 từng có.

## Điều đã ghi vào prereg và không được đổi sau khi thấy kết quả

- claim **duy nhất** được phép: *"gradient matched at the output/logit level at initialization"*
- `η = 0.10` — `D2 ≈ 0` chỉ chứng minh targeting **không hữu ích tại `η = 0.10`**
- `Fracture` chỉ dương nhờ future-positive anchor ⇒ chỉ được gọi **temporal association**
- historical frozen label là primary; known-only là **diagnostic only**
- drift / cosine / rank: **diagnostic**, không gate
- **TEST không bao giờ chạm**